# PIPELINE LÀM SẠCH DESCRIPTION SONG NGỮ (P0–P13)

Theo sơ đồ `docs/design/description_pipeline.html` và quyết định của Nhật Anh ngày 02/10/2026 (`CLAUDE.md` mục 5.2, 5.3b). Chạy bằng kernel `.venv` của folder EDA, **Run All**.

**Nguyên tắc:**
- Không sửa snapshot.
- Mỗi khối ghi ra file riêng trong `data/processed/desc/`: `pN_cap.parquet` (cặp thuộc tính) và `pN_gia_tri.parquet` (từng giá trị).
- Lỗi có quy luật thì **sửa**. Cặp không sửa được thì gắn **mã lý do** (`ly_do`) và không dùng cho train, nhưng **không xoá**.
- Mọi thay đổi ghi vào `log_thay_doi.parquet`, gồm bước, giá trị trước và giá trị sau.

**Cổng duyệt** (thư mục `data/processed/desc/duyet/`):
- Notebook tạo file ứng viên **một lần**. Lần chạy sau nó **đọc lại**, không ghi đè phần đã điền. Mục mới phát sinh được ghi ra file `_bo_sung.csv`.
- Cổng chưa duyệt thì khối đó vẫn chạy bằng **giá trị tạm**. Bảng trạng thái ở cuối notebook cho biết cổng nào còn chờ.
- Dữ liệu chỉ là **chính thức** khi mọi cổng đạt.
- File mẫu kiểm (`*_mau_v1.csv`, `*_mau_v2.csv`) do notebook bốc ngẫu nhiên, seed cố định. Anh điền cột `ket_qua` = `DUNG` hoặc `SAI`.
- Bảng nguồn đổi (sửa luật, Claude điền gợi ý) thì bốc mẫu ở **phiên mới** (`_v2`). File phiên cũ không còn được đọc; anh xoá tay khi muốn:
  `P4_vai_tro_mau_v1.csv`, `P5_danh_so_mau_v1.csv`, `P5_danh_so_le_mau_v1.csv`, `P6_ten_mau_mau_v1.csv`, `P7_gia_tri_mau_mau_v1.csv`, `P13_mau_cuoi_mau_v1.csv`, `P13_mau_cuoi_mau_v2.csv`, `P7_dia_danh_editted.csv` (thay bằng `P7_dia_danh_v2_editted.csv`), `P7_tu_chung_ten_hang_editted.csv` (thay bằng `_v2`), `P7_ten_hang_mau_v1.csv`, `P7_ten_hang_mau_v2.csv`, `P7_ten_hang_mau_v3.csv`, `P7_ten_hang_mau_v4.csv`, `P10_tang1_editted.csv`, `P10_tang1_v2_editted.csv`, `P10_tang1_v3_editted.csv`, `P10_tang1_v4_editted.csv`, `P10_tang1_v5_editted.csv`, `P10_tang1_v6_editted.csv` (tầng 1 tạo lại ở `P10_tang1_v7_editted.csv` sau khi Claude xong P7).
- Mẫu cuối P13 chỉ được bốc khi mọi cổng khác đã đạt (`P13_mau_cuoi_mau_v3.csv`).

## P0. Đầu vào, khoá dữ liệu, quyết định và hàm dùng chung

In [ ]:
import hashlib
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pypinyin import lazy_pinyin
from datasketch import MinHash, MinHashLSH

T0 = time.time()


def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "snapshot" / "bilingual_zh_vi.parquet").exists():
            return p
    raise FileNotFoundError("Không thấy data/snapshot/bilingual_zh_vi.parquet: mở notebook trong folder 'e-commerce - EDA'.")


ROOT = find_root()
SNAP = ROOT / "data" / "snapshot" / "bilingual_zh_vi.parquet"
EDA_DESC = ROOT / "data" / "eda" / "description"
TITLE_AUDIT = ROOT / "data" / "eda" / "title_audit"
OUT = ROOT / "data" / "processed" / "desc"
DUYET = OUT / "duyet"
OUT.mkdir(parents=True, exist_ok=True)
DUYET.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA256 = "B4E5CE75D5AE8AF523235B5C23F5C7289E405A8230D879F11136AFC938135A29"
assert hashlib.sha256(SNAP.read_bytes()).hexdigest().upper() == EXPECTED_SHA256, "SHA256 snapshot đã đổi: dừng lại."
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

QUYET_DINH = {  # Nhật Anh chốt 02/10/2026 (CLAUDE.md 5.3b)
    "sua_hay_loai": "sửa lỗi có quy luật, còn lại gắn mã lý do và không dùng cho train",
    "llm": "chỉ gợi ý (Claude Code cho từ điển và tầng 1; Qwen trên Colab cho tầng 2), người duyệt",
    "dau_ra": "bảng cặp sạch + description_zh/vi dựng lại",
    "thuoc_tinh_giao_dich": "giữ, gắn nhãn vai trò",
    "ten_hang": "pinyin (loại trừ từ chung, đọc 200 mẫu)",
    "thap_phan": "dấu chấm; đổi số có 1–2 chữ số sau dấu khi bản Trung có đúng a.b (02/10: bỏ điều kiện có đơn vị)",
    "ky_hieu": "không xoá; bản Việt 【】 → [ ], giữ ~ | *",
    "chia_tap": "hoãn; chỉ gắn nhom_trung",
    "tieng_anh": "bỏ qua; Quang xử lý sau (P7.6 tắt)",
    "tang2": "phương án A: cặp đuôi dài hiếm gắn T2_CHUA_KIEM, không vào train (Qwen không đạt)",
}
for k, v in QUYET_DINH.items():
    print(f"  {k}: {v}")

# Claude đã điền xong cột gợi ý của bảng nào (đổi thành True sau mỗi phiên Claude Code).
# Chưa xong thì chưa bốc mẫu kiểm của bảng đó, vì mẫu phải bốc từ bảng đã chốt.
CLAUDE_XONG = {"P4": True, "P6": True, "P7": True, "P10_tang1": True}   # P4: 12/300 đổi vai trò; P6: 86/300; P7: 264/914 gợi ý (02/10); P10 tầng 1: 1144 SAI / 3996 (03/10)
print("Claude đã điền xong:", CLAUDE_XONG)
XU_LY_TIENG_ANH = False   # Nhật Anh 02/10: bỏ qua các bước xử lý tiếng Anh, Quang làm sau
DUNG_QWEN = False         # Nhật Anh 03/10 chọn phương án A: Qwen không đạt, tầng 2 gắn mã T2_CHUA_KIEM

CJK = re.compile("[\u4e00-\u9fff]")
VI_DAU = re.compile("[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]", re.I)


def has_cjk(s):
    return bool(CJK.search(s or ""))


def so(n):
    return f"{int(n):,}".replace(",", ".")


def tl(a, b, nd=1):
    return (f"{100.0 * a / b:.{nd}f}%" if b else "0%").replace(".", ",")


def rel(p):
    return Path(p).relative_to(ROOT).as_posix()


def ket_luan(*dong):
    print("\nKẾT LUẬN:")
    for d in dong:
        print("  -", d)


LOG = []      # nhật ký thay đổi
GATES = []    # trạng thái cổng duyệt


def ghi_log(buoc, cot, frame, truoc, sau):
    m = (truoc != sau).values
    if m.any():
        LOG.append(pd.DataFrame({"buoc": buoc, "cot": cot, "pair_id": frame["pair_id"].values[m],
                                 "k": frame["k"].values[m] if "k" in frame else -1,
                                 "truoc": np.asarray(truoc)[m], "sau": np.asarray(sau)[m]}))
    return int(m.sum())


def them_ly_do(pair_ids, code):
    m = P.pair_id.isin(set(pair_ids)) & ~P.ly_do.str.contains(code + ";", regex=False)
    P.loc[m, "ly_do"] = P.loc[m, "ly_do"] + code + ";"
    return int(m.sum())


def them_nhan(pair_ids, code):
    m = P.pair_id.isin(set(pair_ids)) & ~P.nhan.str.contains(code + ";", regex=False)
    P.loc[m, "nhan"] = P.loc[m, "nhan"] + code + ";"
    return int(m.sum())


def cong(ten, viec, trang_thai, chi_tiet=""):
    GATES[:] = [g for g in GATES if g["cong"] != ten]
    GATES.append({"cong": ten, "viec": viec, "trang_thai": trang_thai, "chi_tiet": chi_tiet})
    print(f"[CỔNG {ten}] {trang_thai}: {chi_tiet}")


def file_duyet(ten, frame, khoa):
    """Tạo file duyệt một lần; lần sau đọc lại (không ghi đè).
    Mục mới phát sinh → nối vào *_bo_sung.csv (giữ phần anh đã điền ở đó) và vẫn được áp bằng gợi ý tạm, chờ duyệt."""
    path = DUYET / ten
    if not path.exists():
        frame.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"Tạo mới {rel(path)} ({so(len(frame))} dòng)")
        return frame.astype(str).replace("nan", "")
    cu = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    p2 = path.with_name(path.stem + "_bo_sung.csv")
    bs = pd.read_csv(p2, dtype=str, keep_default_na=False, encoding="utf-8-sig") if p2.exists() else cu.iloc[:0]
    k_da = set(pd.concat([cu, bs])[khoa].astype(str).agg("\x1f".join, axis=1)) if len(cu) + len(bs) else set()
    k_moi = frame[khoa].astype(str).agg("\x1f".join, axis=1)
    them = frame[~k_moi.isin(k_da)].astype(str).replace("nan", "")
    if len(them):
        bs = pd.concat([bs, them], ignore_index=True)
        bs.to_csv(p2, index=False, encoding="utf-8-sig")
        print(f"  {so(len(them))} mục mới chưa có trong file duyệt → nối vào {rel(p2)} (áp tạm, chờ duyệt)")
    print(f"Đọc lại {rel(path)} ({so(len(cu))} dòng, giữ nguyên phần đã điền)" + (f" + {so(len(bs))} dòng bổ sung" if len(bs) else ""))
    return pd.concat([cu, bs], ignore_index=True).fillna("")


def mau_kiem(ten, frame, n, cot, trong_so=None, cho_phep_sai=0, viec="", phien="v1"):
    """Mẫu kiểm ngẫu nhiên (seed cố định). Anh điền ket_qua = DUNG/SAI. Trả về trạng thái cổng.
    Bảng nguồn đổi (sửa luật, Claude điền gợi ý) thì tăng `phien` để bốc mẫu mới; file phiên cũ không dùng nữa."""
    path = DUYET / f"{ten}_mau_{phien}.csv"
    if not path.exists():
        if len(frame) == 0:
            cong(ten, viec, "khong_can", "không có mục nào")
            return "khong_can"
        k = min(n, len(frame))
        rng = np.random.default_rng(SEED)
        if trong_so is None:
            chon_i = rng.choice(len(frame), size=k, replace=False)
        else:   # bốc có trọng số, không hoàn lại (Efraimidis–Spirakis): khoá = u^(1/w), lấy k khoá lớn nhất
            w = trong_so.loc[frame.index].astype(float).clip(lower=1e-9).values
            chon_i = np.argsort(rng.random(len(frame)) ** (1.0 / w))[-k:]
        mau = frame.iloc[np.sort(chon_i)][cot].copy()
        mau["ket_qua"] = ""
        mau["ghi_chu"] = ""
        mau.to_csv(path, index=False, encoding="utf-8-sig")
    m = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    da = (m.ket_qua.str.strip() != "").sum()
    sai = m.ket_qua.str.strip().str.upper().isin(["SAI", "S"]).sum()
    if da < len(m):
        tt, ct = "cho_kiem", f"{da}/{len(m)} mẫu đã kiểm ({rel(path)})"
    elif sai <= cho_phep_sai:
        tt, ct = "dat", f"{len(m)} mẫu, {sai} sai" + (f"; tỉ lệ lỗi < ~{100 * 3 / len(m):.0f}% (95%)" if sai == 0 else "")
    else:
        tt, ct = "khong_dat", f"{len(m)} mẫu, {sai} sai: sửa luật rồi bốc mẫu mới (đổi tên file thành _v2)"
    cong(ten, viec, tt, ct)
    return tt


def luu(buoc):
    P.to_parquet(OUT / f"{buoc}_cap.parquet", compression="zstd")
    V.to_parquet(OUT / f"{buoc}_gia_tri.parquet", compression="zstd")
    print(f"Đã lưu data/processed/desc/{buoc}_cap.parquet, {buoc}_gia_tri.parquet")


print("Thư mục ra:", rel(OUT))

## P1. Tách và ghép cặp thuộc tính (DS01, DS02, DS25)

- Ghép zh–vi theo thứ tự đoạn trong `description_vi`. Không ghép theo chỉ số của `attributes_vi`, vì 99,8% dòng có thứ tự khác.
- 6 dòng không ghép được: gắn mã `DS02` ở cấp sản phẩm.
- Cặp trùng y hệt trong cùng sản phẩm: gắn `DS25`, chỉ giữ một.
- Từ đây làm việc trên 2 bảng:
  - `P`: mỗi dòng là 1 cặp thuộc tính.
  - `V`: mỗi dòng là 1 giá trị trong cặp.

In [ ]:
df = pd.read_parquet(SNAP).reset_index(drop=True)
for c in ["description_zh", "description_vi"]:
    df[c] = df[c].fillna("")


def seg_text(d):
    return f"{d['name']}: {', '.join(d['values'])}"


def recover(desc, attrs):
    segs = [seg_text(d) for d in attrs]
    used = [False] * len(segs)
    pos, order = 0, []
    while pos < len(desc):
        cand = [i for i, s in enumerate(segs) if not used[i] and desc.startswith(s, pos)
                and (pos + len(s) == len(desc) or desc.startswith("; ", pos + len(s)))]
        if not cand:
            return None
        i = max(cand, key=lambda k: len(segs[k]))
        used[i] = True
        order.append(i)
        pos += len(segs[i]) + 2
    return order if all(used) else None


orders = [recover(s, a) for s, a in zip(df.description_vi, df.attributes_vi)]
rp, rv, pid = [], [], 0
for i, (az, av, o) in enumerate(zip(df.attributes_zh, df.attributes_vi, orders)):
    if o is None:
        continue
    for k_, j in enumerate(o):
        rp.append((pid, i, k_, az[k_]["name"], av[j]["name"]))
        for t, (z, v) in enumerate(zip(az[k_]["values"], av[j]["values"])):
            rv.append((pid, t, z, v))
        pid += 1
P = pd.DataFrame(rp, columns=["pair_id", "row", "vi_tri", "name_zh", "name_vi"])
P["product_id"] = df.product_id.values[P.row]
P["category"] = df.category.values[P.row]
P["name_vi_goc"] = P.name_vi
P["ly_do"] = ""
P["nhan"] = ""
V = pd.DataFrame(rv, columns=["pair_id", "k", "v_zh", "v_vi"])
V["v_zh_goc"] = V.v_zh
V["v_vi_goc"] = V.v_vi
V = V.merge(P[["pair_id", "row", "name_zh"]], on="pair_id", how="left")

SP = df[["product_id", "category", "title_zh", "shop", "description_zh", "description_vi"]].copy()
SP["ly_do_sp"] = np.where([o is None for o in orders], "DS02;", "")
print(f"Sản phẩm: {so(len(SP))}; ghép được {so(len(SP) - (SP.ly_do_sp != '').sum())}; DS02: {so((SP.ly_do_sp != '').sum())}")
print(f"Cặp thuộc tính: {so(len(P))}; giá trị: {so(len(V))}")

sig = V.groupby("pair_id").apply(lambda g: "\x1f".join(g.v_zh) + "\x1e" + "\x1f".join(g.v_vi), include_groups=False)
P["_sig"] = P.pair_id.map(sig).fillna("")
dup = P.duplicated(["row", "name_zh", "_sig"], keep="first")
n25 = them_ly_do(P.loc[dup, "pair_id"], "DS25")
P = P.drop(columns="_sig")
luu("p1")
ket_luan(f"{so(len(P))} cặp, {so(len(V))} giá trị từ {so(len(SP) - (SP.ly_do_sp != '').sum())} sản phẩm; 6 sản phẩm DS02 không vào output.",
         f"DS25: {so(n25)} cặp trùng y hệt trong cùng sản phẩm bị gắn mã (giữ 1 bản).")

## P2. Làm sạch ký tự (DS24)

**Áp tự động trên bản Việt** (tên và giá trị):
- Bỏ ký tự vùng riêng (PUA).
- 【】 thành [ ], theo quyết định D16.
- Dấu câu Trung thành dấu ASCII, nhưng chỉ khi bản Việt đã dịch (không còn chữ Hán) và khác bản Trung.
- `"; "` thành `", "`, để không lẫn với dấu phân cách khi dựng lại description.
- Gộp khoảng trắng, bỏ khoảng trắng thừa trong ngoặc và trước dấu phẩy.

**Cổng `P2_sua_tay`** (5 giá trị):
- Giá trị có ký tự hỏng "�" hoặc chữ Hàn: Claude điền gợi ý sẵn.
- Anh ghi `OK`, hoặc ghi bản sửa khác, vào cột `anh_duyet`.
- Chưa duyệt thì cặp đó bị gắn `DS24`.

In [ ]:
PUA = re.compile("[\ue000-\uf8ff]+")
PUNC = str.maketrans({"，": ", ", "。": ". ", "、": ", ", "；": ", ", "：": ": ", "（": "(", "）": ")", "！": "!", "？": "?", "“": '"', "”": '"'})


def sach_ky_tu(s, z):
    s2 = PUA.sub(" ", s).replace("【", "[").replace("】", "]")
    if not has_cjk(s2) and s2 != z:
        s2 = s2.translate(PUNC)
    s2 = s2.replace("; ", ", ")
    s2 = re.sub(r"\s+", " ", s2).strip()
    s2 = re.sub(r"([(\[])\s+", r"\1", s2)
    s2 = re.sub(r"\s+([)\],])", r"\1", s2)
    return s2


truoc = V.v_vi.copy()
V["v_vi"] = [sach_ky_tu(v, z) for v, z in zip(V.v_vi, V.v_zh)]
n_v = ghi_log("P2", "v_vi", V, truoc, V.v_vi)
truoc = P.name_vi.copy()
P["name_vi"] = [sach_ky_tu(v, z) for v, z in zip(P.name_vi, P.name_zh)]
n_n = ghi_log("P2", "name_vi", P, truoc, P.name_vi)

HONG = re.compile("[\ufffd\uac00-\ud7af]")
GOI_Y_P2 = {"意境-橙": "Ý cảnh - Cam", "德绒": "Vải nhung Đức", "崂韵": "Laoyun"}
h = V[V.v_vi.map(lambda s: bool(HONG.search(s)))]
bang = pd.DataFrame({"pair_id": h.pair_id.astype(str), "k": h.k.astype(str), "product_id": P.set_index("pair_id").loc[h.pair_id, "product_id"].values,
                     "ten_zh": h.name_zh, "gia_tri_zh": h.v_zh, "gia_tri_vi": h.v_vi,
                     "goi_y_claude": h.v_zh.map(GOI_Y_P2).fillna(""), "anh_duyet": "", "anh_ghi_chu": ""})
bang = file_duyet("P2_sua_tay_editted.csv", bang, ["pair_id", "k"])
n_sua = 0
for r in bang.itertuples():
    quyet = r.anh_duyet.strip()
    moi = r.goi_y_claude if quyet.upper() == "OK" else quyet
    if moi:
        m = (V.pair_id == int(r.pair_id)) & (V.k == int(r.k))
        ghi_log("P2_tay", "v_vi", V[m], V.loc[m, "v_vi"], pd.Series(moi, index=V.loc[m].index))
        V.loc[m, "v_vi"] = moi
        n_sua += 1
con = V[V.v_vi.map(lambda s: bool(HONG.search(s)))]
them_ly_do(con.pair_id, "DS24")
cong("P2_sua_tay", "5 giá trị ký tự hỏng/chữ Hàn", "dat" if len(con) == 0 else "cho_duyet",
     f"{len(bang)} mục, đã áp {n_sua}; còn {len(con)} giá trị gắn DS24")
luu("p2")
ket_luan(f"Đổi {so(n_v)} giá trị và {so(n_n)} tên tiếng Việt (ký tự vùng riêng, 【】→[ ], dấu câu Trung, khoảng trắng, '; ').",
         f"Ký tự hỏng/chữ Hàn: {len(bang)} giá trị; còn {len(con)} chưa sửa (gắn DS24).")

## P3. Lọc thông tin cá nhân (DS12): mặc định thận trọng, không cần duyệt

- Bỏ hẳn các cặp liên hệ: tên thuộc tính có chứa 联系方式, 电话, 微信 hoặc 联系人. Gắn mã `PII`.
- Các giá trị còn lại thì che bằng nhãn **ở cả hai phía**:
  - số di động Trung Quốc → `[电话]` / `[SĐT]`;
  - số máy bàn có gạch nối;
  - URL → `[URL]`;
  - email → `[EMAIL]`;
  - WeChat/QQ kèm ID → `[联系方式]` / `[LIÊN HỆ]`.
- Cặp tên/địa chỉ nhà sản xuất (生产厂家, 产品厂家厂址, 产品厂家厂名, 生产企业, 厂址, 厂名, 生产地址, 公司, 制造商...) gắn mã `PII_NSX` (charter §6; chốt 02/10).
- Bản công bố bỏ cột `shop`.
- Bước này phải chạy **trước** mọi bước gửi dữ liệu ra LLM.

In [ ]:
TEN_LIEN_HE = re.compile("联系方式|电话|微信|联系人")
m_lh = P.name_zh.map(lambda s: bool(TEN_LIEN_HE.search(s)))
n_lh = them_ly_do(P.loc[m_lh, "pair_id"], "PII")
# Tên / địa chỉ nhà sản xuất (charter §6: bỏ tên shop và địa chỉ trước khi công bố). Nhật Anh chốt 02/10.
# Không gồm 是否源头厂家 (câu hỏi có/không) hay 厂家直销.
TEN_NSX = re.compile(r"^(生产厂家|生产厂商|产品厂家厂址|产品厂家厂名|生产企业|厂址|厂名|生产地址|公司|制造商|生产商|企业名称|经销商|供货商|工厂)$")   # 03/10: thêm 生产厂商, 供货商, 工厂 (29 cặp tên công ty còn lọt)
m_nsx = P.name_zh.map(lambda s: bool(TEN_NSX.match(s.strip())))
n_nsx = them_ly_do(P.loc[m_nsx, "pair_id"], "PII_NSX")
MAU_PII = [
    (re.compile(r"(?<!\d)1[3-9]\d{9}(?!\d)"), "[电话]", "[SĐT]"),
    (re.compile(r"(?<!\d)0\d{2,3}-\d{7,8}(?!\d)"), "[电话]", "[SĐT]"),
    (re.compile(r"https?://\S+|www\.\S+", re.I), "[URL]", "[URL]"),
    (re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"), "[EMAIL]", "[EMAIL]"),
    (re.compile(r"(?:微信|VX|vx|wechat|QQ)\s*[:：号]?\s*[A-Za-z0-9_-]{5,}", re.I), "[联系方式]", "[LIÊN HỆ]"),
]


def che(s, phia):
    for rx, tz, tv in MAU_PII:
        s = rx.sub(tz if phia == "zh" else tv, s)
    return s


tz, tv = V.v_zh.copy(), V.v_vi.copy()
V["v_zh"] = [che(s, "zh") for s in V.v_zh]
V["v_vi"] = [che(s, "vi") for s in V.v_vi]
n_z = ghi_log("P3", "v_zh", V, tz, V.v_zh)
n_vv = ghi_log("P3", "v_vi", V, tv, V.v_vi)
cong("P3_PII", "che thông tin cá nhân", "mac_dinh", "không cần duyệt (hành động an toàn)")
luu("p3")
ket_luan(f"Loại {so(n_lh)} cặp liên hệ (mã PII) và {so(n_nsx)} cặp tên/địa chỉ nhà sản xuất (mã PII_NSX); che {so(n_z)} giá trị zh và {so(n_vv)} giá trị vi.",
         "Bản công bố sau này bỏ cột `shop` (tên shop); file nội bộ vẫn giữ để gộp nhom_trung.")

## P4. Gắn nhãn vai trò và giá trị rỗng nghĩa (DS21)

**Vai trò:**
- Lấy từ `data/eda/description/desc_vai_tro_thuoc_tinh_editted.csv`. Ưu tiên `anh_duyet` (ghi tên vai trò, hoặc `OK` = đồng ý gợi ý), rồi `goi_y_claude`, rồi `vai_tro_goi_y` (luật của EDA).
- Vai trò hợp lệ: `mo_ta_hang`, `giao_dich_1688`, `lua_chon_sku`, `ma_giay_to`, `thuong_hieu`.
- Tên ngoài bảng thì dùng luật (luật mã xét trước luật giao dịch).
- **Cổng `P4_vai_tro_da_sua`:** anh đọc hết các tên Claude đổi vai trò (lý do ở `ly_do_claude`).
- **Cổng `P4_vai_tro`:** 30 mẫu theo trọng số số cặp trong các tên còn lại, bốc sau khi Claude soát xong (`_mau_v2`).

**Giá trị rỗng nghĩa:**
- Dịch về một cách chuẩn: 其他 → "Khác", 无 → "Không có", 见包装 → "Xem bao bì", 咨询客服 → "Liên hệ CSKH"...
- Gắn nhãn `it_thong_tin`.
- Không áp cho 品牌, vì tên hãng xử lý riêng ở P7.
- Cặp mà mọi giá trị chỉ là ký hiệu (`.`, `/`, `-`, `null`) thì gắn mã `RONG`.

In [ ]:
ROLE_RULES = [   # xét theo thứ tự: luật mã trước, để 专利号 / 质检报告编号 là mã chứ không phải giao dịch
    ("ma_giay_to", re.compile(r"货号|^型号$|编号|条码|条形码|许可证|备案|标准号|批准文号|注册证|证号|准字|专利号|认证码")),
    ("giao_dich_1688", re.compile(r"跨境|下游平台|销售地区|货源|加工定制|库存|IP授权|专利|外贸|代发|发货|出货|自有品牌|是否进口|贸易属性|纳税人|价格段|淘货|加印|订制|定制|箱装数量|装箱数量|质检|AQL|产品认证|图片实拍|起批|起订")),
    ("thuong_hieu", re.compile(r"^品牌$")),
]
VAI_HOP_LE = {"mo_ta_hang", "giao_dich_1688", "lua_chon_sku", "ma_giay_to", "thuong_hieu"}
nv_med = V.groupby(["name_zh", "pair_id"]).size().groupby("name_zh").median()   # trung vị số giá trị mỗi cặp


def vai_tro_luat(name):
    for role, rx in ROLE_RULES:
        if rx.search(name):
            return role
    return "lua_chon_sku" if nv_med.get(name, 1) >= 3 else "mo_ta_hang"


vt = pd.read_csv(EDA_DESC / "desc_vai_tro_thuoc_tinh_editted.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
for c in ["goi_y_claude", "ly_do_claude"]:
    if c not in vt.columns:
        vt[c] = ""


def chon_vai(r):
    """Ưu tiên: anh_duyet (tên vai trò; 'OK' = đồng ý gợi ý) > goi_y_claude > vai_tro_goi_y."""
    goi_y = r.goi_y_claude.strip() or r.vai_tro_goi_y.strip()
    a = r.anh_duyet.strip()
    if not a or a.upper() == "OK":
        return goi_y
    if a in VAI_HOP_LE:
        return a
    print(f"  CẢNH BÁO: anh_duyet '{a}' của {r.ten_zh} không phải vai trò hợp lệ {sorted(VAI_HOP_LE)}; tạm dùng '{goi_y}'")
    return goi_y


vt["vai_tro_ap_dung"] = [chon_vai(r) for r in vt.itertuples()]
role_map = dict(zip(vt.ten_zh, vt.vai_tro_ap_dung))
P["vai_tro"] = [role_map.get(n) or vai_tro_luat(n) for n in P.name_zh]
chua_duyet_vt = set(vt.loc[vt.anh_duyet.str.strip() == "", "ten_zh"]) | (set(P.name_zh) - set(vt.ten_zh))
them_nhan(P.loc[P.name_zh.isin(chua_duyet_vt), "pair_id"], "vai_tro_tam")
# Cổng: dòng Claude đổi vai trò thì anh đọc hết; phần còn lại kiểm 30 mẫu theo trọng số (bốc sau khi Claude soát xong)
sua4 = vt.goi_y_claude.str.strip() != ""
ds4 = vt[sua4]
chua4 = (ds4.anh_duyet.str.strip() == "").sum()
cong("P4_vai_tro_da_sua", "vai trò Claude đổi", "dat" if chua4 == 0 else "cho_duyet",
     f"{len(ds4)} tên có goi_y_claude, còn {chua4} chưa có anh_duyet ({rel(EDA_DESC / 'desc_vai_tro_thuoc_tinh_editted.csv')})")
cl4 = vt[~sua4].copy()
cl4["_w"] = pd.to_numeric(cl4.so_cap, errors="coerce").fillna(1)
if CLAUDE_XONG["P4"]:
    mau_kiem("P4_vai_tro", cl4, 30, ["ten_zh", "ten_vi_pho_bien", "gia_tri_zh_pho_bien", "vai_tro_ap_dung"], trong_so=cl4["_w"],
             viec="vai trò giữ theo luật", phien="v2")
else:
    cong("P4_vai_tro", "vai trò giữ theo luật", "cho_claude", "chờ Claude soát bảng vai trò rồi mới bốc 30 mẫu")
display(P.vai_tro.value_counts().rename("so_cap").to_frame().T)
# Giá trị là MÃ thật (chép nguyên ở P8, được giữ chữ Hán):
#   (vai trò mã/giấy tờ VÀ giá trị Trung không có chữ Hán) HOẶC (tên là số giấy phép/đăng ký VÀ giá trị có >= 4 chữ số liền nhau).
#   Vế sau không phụ thuộc vai trò tạm, vì vd 特殊用途化妆品证号 (国妆特字G20191510) đang có vai trò tạm mo_ta_hang.
MA_DANG_KY = re.compile("许可证|备案|批准文号|注册证|标准号|证号|准字")
V["la_ma"] = ((V.pair_id.map(P.set_index("pair_id").vai_tro).eq("ma_giay_to") & ~V.v_zh.map(has_cjk))
              | (V.name_zh.map(lambda s: bool(MA_DANG_KY.search(s))) & V.v_zh.str.contains(r"\d{4,}", regex=True)))
print(f"Giá trị là mã thật (sẽ chép nguyên ở P8): {so(V.la_ma.sum())}")

RONG_NGHIA = {"其他": "Khác", "其它": "Khác", "无": "Không có", "暂无": "Chưa có", "见包装": "Xem bao bì", "详见包装": "Xem bao bì",
              "咨询客服": "Liên hệ CSKH", "详询客服": "Liên hệ CSKH", "联系客服": "Liên hệ CSKH", "如图": "Như hình", "见图": "Như hình",
              "见详情": "Xem chi tiết", "查看详情": "Xem chi tiết", "详情页": "Xem chi tiết", "以实物为准": "Theo sản phẩm thực tế", "不详": "Không rõ"}
KY_HIEU_RONG = {".", "/", "-", "--", "null", "NULL", "—", "..."}
m = V.v_zh.str.strip().isin(RONG_NGHIA.keys()) & (V.name_zh != "品牌")
truoc = V.v_vi.copy()
V.loc[m, "v_vi"] = V.loc[m, "v_zh"].str.strip().map(RONG_NGHIA)
n_rn = ghi_log("P4", "v_vi", V, truoc, V.v_vi)
tat_ca = V.assign(_r=V.v_zh.str.strip().isin(RONG_NGHIA.keys()), _k=V.v_zh.str.strip().isin(KY_HIEU_RONG)).groupby("pair_id")[["_r", "_k"]].all()
them_nhan(tat_ca.index[tat_ca._r], "it_thong_tin")
n_rong = them_ly_do(tat_ca.index[tat_ca._k], "RONG")
luu("p4")
ket_luan(f"Vai trò: {so((P.vai_tro == 'giao_dich_1688').sum())} cặp giao dịch 1688 (giữ, có nhãn); {so(P.nhan.str.contains('vai_tro_tam').sum())} cặp có vai trò tạm (chưa duyệt).",
         f"Giá trị rỗng nghĩa: dịch chuẩn {so(n_rn)} giá trị; {so(len(tat_ca[tat_ca._r]))} cặp gắn it_thong_tin; {so(n_rong)} cặp chỉ có ký hiệu (mã RONG).")

## P5. Gỡ lỗi đánh số (DS06, DS06B)

**Luật v2:** bỏ 1–2 chữ số cuối khi thoả cả ba điều kiện:
1. Bỏ đi thì tập số khớp bản Trung.
2. Số đó không phải chữ số Hán có trong bản Trung (二级 → "Cấp độ 2" là dịch đúng).
3. Phần còn lại trùng một bản dịch khác trong cùng sản phẩm.

**Cổng:**
- **`P5_danh_so`:** 30 mẫu ngẫu nhiên trong các giá trị đã sửa.
- **`P5_danh_so_le`:** dành cho nhóm "đánh số lẻ" (không thấy bản dịch trùng).
  - Gồm 100 mẫu; chỉ áp cho cả nhóm khi số mẫu sai không quá 5 (đúng từ 95%).
  - Chưa đạt thì giữ nguyên số; P8 sẽ bắt các cặp này là lệch số.

In [ ]:
CN_NUM = {"一": 1, "二": 2, "两": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8, "九": 9, "十": 10, "双": 2, "单": 1}
CN_CHU_SO = {"一": 1, "二": 2, "两": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8, "九": 9}
DIG = re.compile(r"\d+")


def so_han_ghep(run):
    """十一 → 11, 二十七 → 27, 一百二十 → 120."""
    tong, cur = 0, 0
    for ch in run:
        if ch in CN_CHU_SO:
            cur = CN_CHU_SO[ch]
        elif ch == "十":
            tong, cur = tong + (cur or 1) * 10, 0
        elif ch == "百":
            tong, cur = tong + (cur or 1) * 100, 0
    return tong + cur


def goc_truoc_so(z, v):
    nz = sorted(DIG.findall(z))
    # số Hán có trong bản Trung: từng chữ (双 = 2) và cả số ghép (图十一 = 11, không được bỏ '11' của 'Hình 11')
    cn = {CN_NUM[c] for c in z if c in CN_NUM} | {so_han_ghep(r) for r in re.findall("[一二两三四五六七八九十百]+", z)}
    for L in (1, 2):
        if len(v) > L and v[-L:].isascii() and v[-L:].isdigit() and int(v[-L:]) not in cn:
            b = v[:-L].rstrip(".").rstrip()
            if b and sorted(DIG.findall(b)) == nz:
                return b
    return ""


V["_goc"] = [goc_truoc_so(z, v) for z, v in zip(V.v_zh, V.v_vi)]
V["_khoa"] = [(g or v).lower().strip() for g, v in zip(V._goc, V.v_vi)]
V["_trung"] = V.groupby(["row", "_khoa"])._khoa.transform("size")
m_v2 = (V._goc != "") & (V._trung >= 2)
m_le = (V._goc != "") & (V._trung < 2)
truoc = V.v_vi.copy()
V.loc[m_v2, "v_vi"] = V.loc[m_v2, "_goc"]
n_v2 = ghi_log("P5", "v_vi", V, truoc, V.v_vi)
sua = V[m_v2].assign(truoc=truoc[m_v2], sau=V.loc[m_v2, "v_vi"])
mau_kiem("P5_danh_so", sua, 30, ["row", "name_zh", "v_zh", "truoc", "sau"], viec="bỏ số đuôi luật v2", phien="v2")   # v2: sau khi sửa số Hán ghép (02/10)

le = V[m_le].assign(truoc=V.loc[m_le, "v_vi"], sau=V.loc[m_le, "_goc"])
tt_le = mau_kiem("P5_danh_so_le", le, 100, ["row", "name_zh", "v_zh", "truoc", "sau"], cho_phep_sai=5, viec="số đuôi lẻ (DS06B)",
                  phien="v2")   # v2: v1 có 1 lỗi số Hán ghép (款式十一), đã sửa luật
n_le = 0
if tt_le == "dat":
    truoc = V.v_vi.copy()
    V.loc[m_le, "v_vi"] = V.loc[m_le, "_goc"]
    n_le = ghi_log("P5_le", "v_vi", V, truoc, V.v_vi)
V = V.drop(columns=["_goc", "_khoa", "_trung"])
luu("p5")
ket_luan(f"Luật v2 bỏ số đuôi ở {so(n_v2)} giá trị.",
         f"Nhóm đánh số lẻ: {so(m_le.sum())} giá trị; " + (f"đã áp {so(n_le)}." if n_le else "CHƯA áp (chờ 100 mẫu đạt)."))

## P6. Từ điển tên thuộc tính (DS15, DS16, DS05, DS07 tên)

File `duyet/P6_ten_thuoc_tinh_editted.csv` gồm 300 tên phổ biến nhất, cộng các tên sai nghĩa đã biết. Mỗi dòng có:
- `cach_dich_da_so`: cách dịch đa số, đã gộp khác biệt chữ hoa.
- `loi_da_biet`: lỗi đã biết, kèm nghĩa đúng.
- `ten_chuan_tam`: tên dùng tạm. Nếu tên sai nghĩa thì đây là nghĩa đúng, còn lại là cách dịch đa số.
- `da_sua` = 1 khi tên chuẩn khác cách dịch đa số.
- `goi_y_claude`: em điền trong các phiên Claude Code.
- `anh_duyet`: anh ghi `OK`, hoặc ghi tên khác.

**Thứ tự ưu tiên khi áp:**
1. `anh_duyet`, nếu anh ghi một tên khác `OK`;
2. `goi_y_claude`;
3. `ten_chuan_tam`.

**Cổng `P6_ten`** đạt khi:
- mọi dòng đã sửa đều có `anh_duyet`. "Đã sửa" được tính lại mỗi lần chạy: `da_sua` = 1, hoặc `goi_y_claude` khác cách dịch đa số;
- 30 mẫu theo trọng số trong các dòng còn lại không có lỗi. Mẫu chỉ được bốc sau khi Claude điền xong (`CLAUDE_XONG["P6"]` ở P0), vào file `_mau_v2`.

Tên ngoài bảng thì chỉ viết hoa chữ đầu.

In [ ]:
TEN_SAI = {
    "毛重": ("Trọng lượng cả bì", "dịch thành 'tịnh' (ngược nghĩa) / 'lông'"), "容量": ("Dung tích", "'công suất' / 'năng lực'"),
    "皮质特征": ("Đặc điểm da", "'vỏ não'"), "适用场景": ("Dịp sử dụng", "'kịch bản'"), "使用场景": ("Tình huống sử dụng", "'kịch bản'"),
    "上市年份季节": ("Năm/mùa ra mắt", "'niêm yết'"), "上市年份季节（上市时间）": ("Năm/mùa ra mắt (thời gian ra mắt)", "'niêm yết'"),
    "上市年份/季节": ("Năm/mùa ra mắt", "'niêm yết'"), "净含量": ("Khối lượng tịnh", "'nội dung'"),
    "净含量（规格）": ("Khối lượng tịnh (quy cách)", "'nội dung'"), "主面料成分含量": ("Tỉ lệ thành phần vải chính", "'nội dung'"),
    "主面料成分的含量": ("Tỉ lệ thành phần vải chính", "'nội dung'"), "适合人群": ("Đối tượng phù hợp", "'đám đông'"),
    "适用人群": ("Đối tượng sử dụng", "'đám đông'"), "安全等级": ("Cấp độ an toàn", "'bảo mật' / 'an ninh'"),
    "有无夹层": ("Có ngăn phụ không", "'tầng lửng' / 'gác lửng'"), "鞋头形状": ("Hình dạng mũi giày", "'ngón chân'"),
    "鞋跟形状": ("Hình dạng gót giày", "'gót chân'"), "鞋跟高度": ("Chiều cao gót giày", "'gót chân'"), "鞋跟款式": ("Kiểu gót giày", "'gót chân'"),
    "鞋底工艺": ("Công nghệ đế giày", "'duy nhất' (sole)"), "版型": ("Form dáng", "'phiên bản'"), "口径规格": ("Đường kính miệng", "'cỡ nòng'"),
    "面料支数": ("Chi số sợi vải", "'số lượng vải'"), "元素": ("Yếu tố", "'nguyên tố' / 'phần tử'"), "起泡程度": ("Độ tạo bọt", "'phồng rộp'"),
    "热卖月份": ("Tháng bán chạy", "'tháng nóng'"), "图案": ("Hoa văn", "'mô hình'"), "领标": ("Nhãn cổ áo", "'giá thầu' / 'lãnh đạo'"),
    "传输速率": ("Tốc độ truyền", "'chuyển khoản'"), "型号": ("Model", "'người mẫu' / 'mô hình'"), "闭合方式": ("Kiểu đóng", "'đóng cửa'"),
    "口味": ("Hương vị", "'nếm thử'"),
}


def norm(s):
    return s.strip(" ?.:？").lower()


def cap(s):
    return s[:1].upper() + s[1:] if s else s


freq = P.name_zh.value_counts()
names = list(freq.head(300).index) + [n for n in TEN_SAI if n not in set(freq.head(300).index) and n in freq.index]
rows = []
for n in names:
    s = P.loc[P.name_zh == n, "name_vi"]
    vc = s[~s.map(has_cjk)].map(norm).value_counts()
    maj = cap(vc.index[0]) if len(vc) else ""
    loi = ""
    if n in TEN_SAI:
        chuan, loi = TEN_SAI[n][0], "sai nghĩa: " + TEN_SAI[n][1]
    else:
        chuan = maj
        if n.startswith("是否") and re.search(r"^có nên|cho dù|^dù ", maj.lower()):
            loi = "是否 bị dịch thành 'có nên'/'cho dù': cần Claude gợi ý"
        if not maj:
            loi = "chưa có bản dịch"
    rows.append({"ten_zh": n, "so_cap": int(freq[n]), "cach_dich_da_so": maj,
                 "cac_cach_dich": "; ".join(f"{k}: {v}" for k, v in s.map(norm).value_counts().head(5).items()),
                 "loi_da_biet": loi, "ten_chuan_tam": chuan, "da_sua": int(norm(chuan) != norm(maj) or bool(loi)),
                 "goi_y_claude": "", "anh_duyet": "", "anh_ghi_chu": ""})
TD6 = file_duyet("P6_ten_thuoc_tinh_editted.csv", pd.DataFrame(rows), ["ten_zh"])


def chon(r):
    a = r.anh_duyet.strip()
    if a and a.upper() != "OK":
        return a
    return r.goi_y_claude.strip() or r.ten_chuan_tam.strip()


ten_map = {r.ten_zh: chon(r) for r in TD6.itertuples() if chon(r)}
truoc = P.name_vi.copy()
P["name_vi"] = [ten_map.get(z) or cap(v) for z, v in zip(P.name_zh, P.name_vi)]
n6 = ghi_log("P6", "name_vi", P, truoc, P.name_vi)
# Đã sửa = da_sua lúc tạo file, HOẶC Claude gợi ý một tên khác cách dịch đa số (tính lại mỗi lần chạy)
sua6 = (TD6.da_sua.astype(str) == "1") | ((TD6.goi_y_claude.str.strip() != "") & (TD6.goi_y_claude.map(norm) != TD6.cach_dich_da_so.map(norm)))
ds = TD6[sua6]
chua = (ds.anh_duyet.str.strip() == "").sum()
con_lai = TD6[~sua6].copy()
con_lai["_w"] = pd.to_numeric(con_lai.so_cap, errors="coerce").fillna(1)
if CLAUDE_XONG["P6"]:
    tt_m = mau_kiem("P6_ten_mau", con_lai, 30, ["ten_zh", "so_cap", "cach_dich_da_so", "cac_cach_dich"], trong_so=con_lai["_w"],
                    viec="tên giữ cách dịch đa số", phien="v2")
else:
    cong("P6_ten_mau", "tên giữ cách dịch đa số", "cho_claude", "chờ Claude điền goi_y_claude rồi mới bốc 30 mẫu")
cong("P6_ten_da_sua", "tên Claude sửa khác cách dịch đa số", "dat" if chua == 0 else "cho_duyet",
     f"{len(ds)} tên da_sua=1, còn {chua} chưa có anh_duyet ({rel(DUYET / 'P6_ten_thuoc_tinh_editted.csv')})")
luu("p6")
ket_luan(f"Đổi tên ở {so(n6)} cặp; từ điển {len(TD6)} tên, {len(ds)} tên khác cách dịch đa số (anh đọc hết), còn lại kiểm 30 mẫu.",
         f"Claude gợi ý {so((TD6.goi_y_claude.str.strip() != '').sum())} tên (cột goi_y_claude, lý do ở ly_do_claude)."
         if CLAUDE_XONG["P6"] else "Tên '是否...' dịch sai và tên chưa có bản dịch chờ Claude gợi ý (cột goi_y_claude).")

## P7. Từ điển giá trị có điều kiện (DS17, DS18, DS19, DS13, DS09A/B, DS08, DS07 giá trị)

1. **Từ điển giá trị** (`duyet/P7_gia_tri_editted.csv`): các giá trị chữ Hán gặp ≥ 50 lần, cách làm giống P6. Áp theo `gia_tri_zh`, trừ 品牌 và cặp mã/giấy tờ.
2. **Luật theo ngữ cảnh:** 苹果 ở thuộc tính dòng máy → "Apple"; 中 ở kích cỡ → "Vừa", ở độ cứng → "Trung bình".
3. **Lỗi sai nghĩa đã biết nằm trong chuỗi dài hơn:** không sửa được an toàn nên gắn mã `DS17`.
4. **Địa danh** (`duyet/P7_dia_danh_v2_editted.csv`):
   - Giá trị nơi sản xuất chỉ gồm địa danh đã biết thì dựng lại "Thành phố, Tỉnh".
   - Có địa danh nhưng bản Việt sai thì gắn `DS18`.
5. **Tên hãng (品牌):**
   - Từ chung thì dịch theo `duyet/P7_tu_chung_ten_hang_v2_editted.csv`; "other/others" → "Khác".
   - Tên toàn chữ Latin: chép nguyên bản Trung. Có phần Latin thì giữ phần Latin; không có thì chuyển pinyin, giữ chữ số (70迈 → 70mai).
   - Chú thích loại hàng trong ngoặc được dịch nghĩa: 天王（食品）→ "Tianwang (thực phẩm)".
   - Phần chính dài hơn 7 chữ Hán, hoặc là tên công ty (有限公司, 厂, 纺织品...), thì gắn `DS09_dai`.
   - **Cổng `P7_ten_hang`:** anh đọc 200 mẫu; đạt khi sai không quá 10 (5%, chốt 02/10).
6. **Tiếng Anh/pinyin:** **đang TẮT** (quyết định 02/10: Quang xử lý phần tiếng Anh sau; bật lại bằng `XU_LY_TIENG_ANH = True` ở P0). Khi bật: áp các từ THAY/HANVIET của bảng title, chỉ khi từ đó không có trong bản Trung.
7. **Thuật ngữ chung với title** (`duyet/P7_thuat_ngu_editted.csv`): thay các cách dịch khác bằng thuật ngữ chuẩn. Thuật ngữ để trống `chuan` thì không áp.

In [ ]:
# 7.1 Từ điển giá trị
GT_DUNG = {"基础大众": "Phổ thông", "纯色": "Màu trơn", "单根": "Một dây", "双根": "Hai dây", "前系带": "Buộc dây phía trước", "低帮": "Cổ thấp",
           "长爬": "Bộ liền thân", "生活防水": "Chống nước sinh hoạt", "粘胶鞋": "Giày dán keo", "无领标": "Không có nhãn cổ", "有领标": "Có nhãn cổ",
           "韩版": "Kiểu Hàn", "实拍无模特": "Ảnh thật, không có người mẫu", "实拍有模特": "Ảnh thật, có người mẫu", "宽口径": "Miệng rộng",
           "套头": "Chui đầu", "日常搭配": "Phối đồ hằng ngày", "明前": "Minh Tiền", "单鞋": "Giày bệt", "磨毛": "Vải chải lông",
           "是": "Có", "否": "Không", "有": "Có", "通用": "Thông dụng"}
CHUAN_GT = V[V.v_zh.map(has_cjk) & (V.name_zh != "品牌")]
cnt = CHUAN_GT.v_zh.value_counts()
pho_bien = cnt[cnt >= 50].index
rows = []
nhom_gt = {z: g for z, g in CHUAN_GT[CHUAN_GT.v_zh.isin(pho_bien)].groupby("v_zh")}
for z in pho_bien:
    g = nhom_gt[z]
    s = g.v_vi
    s_ok = s[~s.map(has_cjk)]
    vc = s_ok.map(lambda x: x.strip(" .").lower()).value_counts()
    if len(vc):
        maj_n = vc.index[0]
        maj = s_ok[s_ok.map(lambda x: x.strip(" .").lower()) == maj_n].value_counts().index[0].strip(" .")
    else:
        maj = ""
    chuan = GT_DUNG.get(z, maj)
    rows.append({"gia_tri_zh": z, "so_lan": int(cnt[z]), "ten_thuoc_tinh_pho_bien": g.name_zh.value_counts().index[0],
                 "cach_dich_da_so": maj, "cac_cach_dich": "; ".join(f"{k}: {v}" for k, v in vc.head(5).items()),
                 "loi_da_biet": "sai nghĩa (đọc tay)" if z in GT_DUNG and norm(GT_DUNG[z]) != norm(maj) else ("chưa có bản dịch" if not maj else ""),
                 "gia_tri_chuan_tam": chuan, "da_sua": int(norm(chuan) != norm(maj) or not maj),
                 "goi_y_claude": "", "anh_duyet": "", "anh_ghi_chu": ""})
TD7 = file_duyet("P7_gia_tri_editted.csv", pd.DataFrame(rows), ["gia_tri_zh"])


def chon7(r):
    a = r.anh_duyet.strip()
    if a and a.upper() != "OK":
        return a
    return r.goi_y_claude.strip() or r.gia_tri_chuan_tam.strip()


gt_map = {r.gia_tri_zh: chon7(r) for r in TD7.itertuples() if chon7(r)}
la_ma = V.la_ma
m = V.v_zh.isin(gt_map.keys()) & (V.name_zh != "品牌") & ~la_ma
truoc = V.v_vi.copy()
V.loc[m, "v_vi"] = V.loc[m, "v_zh"].map(gt_map)
n71 = ghi_log("P7_tu_dien", "v_vi", V, truoc, V.v_vi)
sua7 = (TD7.da_sua.astype(str) == "1") | ((TD7.goi_y_claude.str.strip() != "") & (TD7.goi_y_claude.map(norm) != TD7.cach_dich_da_so.map(norm)))
ds7 = TD7[sua7]
chua7 = (ds7.anh_duyet.str.strip() == "").sum()
cl7 = TD7[~sua7].copy()
cl7["_w"] = pd.to_numeric(cl7.so_lan, errors="coerce").fillna(1)
if CLAUDE_XONG["P7"]:
    mau_kiem("P7_gia_tri_mau", cl7, 30, ["gia_tri_zh", "so_lan", "cach_dich_da_so", "cac_cach_dich"], trong_so=cl7["_w"],
             viec="giá trị giữ cách dịch đa số", phien="v2")
else:
    cong("P7_gia_tri_mau", "giá trị giữ cách dịch đa số", "cho_claude", "chờ Claude điền goi_y_claude rồi mới bốc 30 mẫu")
cong("P7_gia_tri_da_sua", "giá trị Claude sửa khác cách dịch đa số", "dat" if chua7 == 0 else "cho_duyet",
     f"{len(ds7)} giá trị da_sua=1, còn {chua7} chưa có anh_duyet")

# 7.2 Luật theo ngữ cảnh
NGU_CANH = [({"适用型号", "适用机型", "苹果型号", "规格型号"}, "苹果", "Apple"),
            ({"箱包大小", "包大小", "尺寸"}, "中", "Vừa"), ({"硬度", "鞋底硬度", "防滑性能", "定型性能"}, "中", "Trung bình")]
truoc = V.v_vi.copy()
for ten, z, dung in NGU_CANH:
    mm = V.name_zh.isin(ten) & (V.v_zh == z)
    V.loc[mm, "v_vi"] = dung
n72 = ghi_log("P7_ngu_canh", "v_vi", V, truoc, V.v_vi)

# 7.3 Lỗi sai nghĩa nằm trong chuỗi dài: gắn DS17
GT_SAI = [("鸭屎香", "phân vịt"), ("密码箱", "hộp mật khẩu"), ("雪梨", "sydney"), ("冰岛", "iceland"), ("杰理", "jerry"), ("中科蓝讯", "blue news"),
          ("大红袍", "áo choàng đỏ"), ("基础大众", "volkswagen"), ("长爬", "leo"), ("低帮", "mũ thấp"), ("低帮", "đỉnh thấp"), ("生活防水", "cuộc sống"),
          ("粘胶鞋", "viscose"), ("前系带", "cà vạt"), ("无领标", "không có cổ áo"), ("韩版", "tiếng hàn"), ("纯色", "màu rắn"), ("实拍无模特", "mô hình"),
          ("单根", "gốc"), ("双根", "rễ"), ("宽口径", "cỡ nòng"), ("套头", "mũ trùm"), ("套头", "xe kéo"), ("日常搭配", "đối sánh"),
          ("明前", "nhà minh"), ("明前", "trung thu"), ("明前", "ngày mai"), ("单鞋", "căn hộ"), ("磨毛", "đánh răng"), ("皮质", "vỏ não")]
vl = V.v_vi.str.lower()
m17 = pd.Series(False, index=V.index)
for z, w in GT_SAI:
    m17 |= V.v_zh.str.contains(z, regex=False) & vl.str.contains(w, regex=False)
n17 = them_ly_do(V.loc[m17, "pair_id"], "DS17")

# 7.4 Địa danh
DIA_DANH = {"温州": "Ôn Châu", "台州": "Đài Châu", "义乌": "Nghĩa Ô", "广州": "Quảng Châu", "深圳": "Thâm Quyến", "东莞": "Đông Hoản", "汕头": "Sán Đầu",
            "杭州": "Hàng Châu", "金华": "Kim Hoa", "宁波": "Ninh Ba", "泉州": "Tuyền Châu", "晋江": "Tấn Giang", "莆田": "Bồ Điền", "保定": "Bảo Định",
            "白沟": "Bạch Câu", "揭阳": "Yết Dương", "佛山": "Phật Sơn", "中山": "Trung Sơn", "潮州": "Triều Châu", "湖州": "Hồ Châu", "苏州": "Tô Châu",
            "南通": "Nam Thông", "临沂": "Lâm Nghi", "安溪": "An Khê", "福鼎": "Phúc Đỉnh", "武夷山": "Vũ Di Sơn", "云南": "Vân Nam", "普洱": "Phổ Nhĩ",
            "浙江": "Chiết Giang", "广东": "Quảng Đông", "福建": "Phúc Kiến", "河北": "Hà Bắc", "河南": "Hà Nam", "江苏": "Giang Tô", "山东": "Sơn Đông",
            "安徽": "An Huy", "湖南": "Hồ Nam", "四川": "Tứ Xuyên", "上海": "Thượng Hải", "北京": "Bắc Kinh", "江西": "Giang Tây",
            "中国": "Trung Quốc", "中国大陆": "Trung Quốc đại lục",
            # bổ sung 02/10 sau khi soát 1.544 giá trị không tách được (vd 新余 bị dịch 'Tân Ngu', 徐州 'Tô Châu', 澄海 'Thành Hải')
            "新余": "Tân Dư", "洛阳": "Lạc Dương", "惠州": "Huệ Châu", "焦作": "Tiêu Tác", "望都": "Vọng Đô", "湖北": "Hồ Bắc",
            "广西": "Quảng Tây", "安阳": "An Dương", "澄海": "Trừng Hải", "阳江": "Dương Giang", "瑞安": "Thụy An", "温县": "Huyện Ôn",
            "陕西": "Thiểm Tây", "荆州": "Kinh Châu", "偃师": "Yển Sư", "普宁": "Phổ Ninh", "永康": "Vĩnh Khang", "郑州": "Trịnh Châu",
            "天津": "Thiên Tân", "重庆": "Trùng Khánh", "贵州": "Quý Châu", "张家界": "Trương Gia Giới", "成都": "Thành Đô",
            "徐州": "Từ Châu", "福州": "Phúc Châu", "舞阳": "Vũ Dương", "桂林": "Quế Lâm", "黑龙江": "Hắc Long Giang",
            "内蒙古": "Nội Mông Cổ", "云和": "Vân Hòa", "丽水": "Lệ Thủy", "上饶": "Thượng Nhiêu", "吉林": "Cát Lâm", "合肥": "Hợp Phì",
            "哈尔滨": "Cáp Nhĩ Tân", "温岭": "Ôn Lĩnh", "香港": "Hồng Kông", "台湾": "Đài Loan", "澳门": "Ma Cao",
            "韩国": "Hàn Quốc", "日本": "Nhật Bản", "泰国": "Thái Lan", "越南": "Việt Nam", "印尼": "Indonesia"}
PINYIN_DD = {"温州": "wenzhou", "台州": "taizhou", "义乌": "yiwu", "广州": "guangzhou", "深圳": "shenzhen", "东莞": "dongguan", "汕头": "shantou",
             "杭州": "hangzhou", "金华": "jinhua", "宁波": "ningbo", "泉州": "quanzhou", "晋江": "jinjiang", "莆田": "putian", "保定": "baoding",
             "白沟": "baigou", "揭阳": "jieyang", "佛山": "foshan", "中山": "zhongshan", "潮州": "chaozhou", "湖州": "huzhou", "苏州": "suzhou",
             "南通": "nantong", "临沂": "linyi", "安溪": "anxi", "福鼎": "fuding", "武夷山": "wuyi", "云南": "yunnan", "普洱": "puer"}
PINYIN_DD = {**{k: "".join(lazy_pinyin(k)) for k in DIA_DANH if k not in PINYIN_DD}, **PINYIN_DD}   # bản Việt để pinyin cũng không tính là sai
bdd = pd.DataFrame({"dia_danh_zh": list(DIA_DANH), "han_viet_goi_y": list(DIA_DANH.values()), "anh_duyet": "", "anh_ghi_chu": ""})
bdd = file_duyet("P7_dia_danh_v2_editted.csv", bdd, ["dia_danh_zh"])   # v2: bảng mở rộng 02/10 (v1 chỉ có 43 mục)
dd_map = {r.dia_danh_zh: (r.anh_duyet.strip() if r.anh_duyet.strip() and r.anh_duyet.strip().upper() != "OK" else r.han_viet_goi_y) for r in bdd.itertuples()}
dd_keys = sorted(dd_map, key=len, reverse=True)
TEN_NOI = {"产地", "原产地", "发货地", "产区", "原产国/地区", "产地/城市", "产地-国内"}


def tach_dia_danh(z):
    """'广东省广州市' → ['广东', '广州']. Hậu tố 省/市/区/县/镇 chỉ bỏ khi đứng ngay sau một địa danh đã biết
    (để '温县' vẫn là một mục). Còn phần không nhận ra thì trả None (không dựng lại)."""
    s = re.sub(r"[\s,，/、;；]", "", z)
    out = []
    while s:
        k = next((k for k in dd_keys if s.startswith(k)), None)
        if not k:
            return None
        out.append(k)
        s = s[len(k):]
        if s[:1] in set("省市区县镇"):
            s = s[1:]
    return out


truoc = V.v_vi.copy()
ds18 = []
mnoi = V.name_zh.isin(TEN_NOI)
V["dd_dung_lai"] = False   # giá trị đã dựng lại từ bảng địa danh: không cần đưa vào hàng đợi kiểm nghĩa P10
for i in V.index[mnoi]:
    z, v = V.at[i, "v_zh"], V.at[i, "v_vi"]
    t = tach_dia_danh(z)
    if t:
        V.at[i, "v_vi"] = ", ".join(dd_map[k] for k in reversed(t))
        V.at[i, "dd_dung_lai"] = True
    else:
        for k in dd_keys:
            if k in z and dd_map[k].lower() not in v.lower() and PINYIN_DD.get(k, "@@") not in v.lower():
                ds18.append(V.at[i, "pair_id"])
                break
n18 = them_ly_do(ds18, "DS18")
n74 = ghi_log("P7_dia_danh", "v_vi", V, truoc, V.v_vi)
cong("P7_dia_danh", "bảng địa danh Hán Việt", "dat" if (bdd.anh_duyet.str.strip() != "").all() else "cho_duyet",
     f"{len(bdd)} địa danh, còn {(bdd.anh_duyet.str.strip() == '').sum()} chưa duyệt")

# 7.5 Tên hãng → pinyin
TU_CHUNG = {"其他": "Khác", "其它": "Khác", "无": "Không thương hiệu", "无品牌": "Không thương hiệu", "中性": "Không thương hiệu", "通用": "Thông dụng",
            "厂家": "Nhà sản xuất", "自有品牌": "Thương hiệu riêng", "定制": "Theo yêu cầu", "暂无": "Chưa có", "原创": "Thiết kế gốc",
            "国产": "Hàng nội địa", "其他品牌": "Thương hiệu khác", "见包装": "Xem bao bì", "工厂": "Nhà máy", "源头工厂": "Nhà máy gốc",
            # bổ sung 02/10 sau khi soát 15.834 giá trị 品牌 (vd 自主 bị phiên âm 'Zizhu', 否 thành 'Fou')
            "自主": "Thương hiệu riêng", "自主品牌": "Thương hiệu riêng", "自主研发": "Tự nghiên cứu phát triển", "否": "Không có",
            "默认项": "Mặc định", "其他家": "Khác", "无品牌（无注册商标）": "Không thương hiệu (không có nhãn hiệu đăng ký)",
            "无品牌/无注册商标": "Không thương hiệu (không có nhãn hiệu đăng ký)", "不挂网线下品牌": "Thương hiệu bán trực tiếp, không bán online",
            "工厂自营": "Nhà máy tự bán", "工厂直销": "Nhà máy bán trực tiếp", "工厂货": "Hàng nhà máy", "工厂白牌": "Hàng nhà máy không thương hiệu",
            # bổ sung 02/10 sau khi Claude đọc 200 mẫu (~5% là từ chung/mô tả bị phiên âm) và quét toàn bộ 品牌
            "没有": "Không có", "无其他": "Không có", "无合适": "Không có", "非品牌": "Không thương hiệu", "白牌": "Không thương hiệu",
            "自有": "Thương hiệu riêng", "自建": "Thương hiệu riêng", "自由品牌": "Thương hiệu riêng", "自由产品": "Sản phẩm tự sản xuất",
            "自营": "Tự kinh doanh", "自主生产": "Tự sản xuất", "自检": "Tự kiểm tra", "设计师品牌": "Thương hiệu nhà thiết kế",
            "国内品牌": "Thương hiệu nội địa", "国货品牌": "Thương hiệu nội địa", "国货": "Hàng nội địa", "外贸": "Hàng xuất khẩu",
            "多种品牌": "Nhiều thương hiệu", "多品牌": "Nhiều thương hiệu", "品牌": "Thương hiệu", "其它家": "Khác", "其他的": "Khác",
            "其他.": "Khác", "其他·": "Khác", "厂家直销": "Nhà máy bán trực tiếp", "源头厂家": "Nhà sản xuất gốc",
            "贴牌": "Gia công OEM", "可贴牌": "Nhận gia công OEM", "贴牌代加工": "Gia công OEM", "授权贴牌": "Gia công OEM được ủy quyền",
            "代加工": "Gia công", "可定制": "Nhận đặt theo yêu cầu", "带logo": "Kèm logo", "定制LOGO": "Logo theo yêu cầu",
            "无，接受定制": "Không có, nhận đặt theo yêu cầu", "见全": "Xem đầy đủ", "见描述": "Xem mô tả", "见外包装": "Xem bao bì",
            "详见包装": "Xem bao bì", "详见外包装": "Xem bao bì", "见产品包装": "Xem bao bì", "玩具": "Đồ chơi", "女包": "Túi nữ",
            "童装": "Quần áo trẻ em", "运动鞋": "Giày thể thao", "赛车鞋": "Giày đua", "织带款": "Kiểu ruy băng", "精品箱包": "Túi xách cao cấp",
            "莆田鞋": "Giày Bồ Điền", "莆田运动鞋": "Giày thể thao Bồ Điền", "老北京布鞋": "Giày vải Bắc Kinh", "食用农产品": "Nông sản ăn được",
            "绿茶": "Trà xanh", "普洱茶": "Trà Phổ Nhĩ", "英德红茶": "Hồng trà Anh Đức", "安化黑茶": "Hắc trà An Hóa", "单枞茶": "Trà Đơn Tùng",
            "人参六宝茶": "Trà nhân sâm Lục Bảo", "恩施硒茶": "Trà selen Ân Thi",
            "饭盒": "Hộp cơm", "实力商家": "Nhà bán hàng uy tín", "都匀毛尖": "Trà Mao Tiêm Đô Quân", "中性/Neatral": "Không thương hiệu",
            "中性/neutral": "Không thương hiệu", "咨询客服": "Liên hệ CSKH", "详询客服": "Liên hệ CSKH", "苹果": "Apple",
            "氨基酸洁面乳": "Sữa rửa mặt axit amin", "奶糖": "Kẹo sữa", "收纳盒": "Hộp đựng đồ", "汽车置物袋": "Túi đựng đồ ô tô",
            "时尚通勤腋下包": "Túi đeo nách thời trang đi làm"}
# Tên hãng + hậu tố loại hàng: dịch phần loại hàng, phiên âm phần tên (美凤箱包 → 'Túi xách Meifeng', 小伶玩具 → 'Đồ chơi Xiaoling')
HAU_TO_LOAI = {"汽车用品": "Đồ dùng ô tô", "箱包": "Túi xách", "玩具": "Đồ chơi", "童装": "Thời trang trẻ em", "包装": "Bao bì",
               "茶业": "Trà", "鞋业": "Giày", "茶叶": "Trà", "服饰": "Thời trang", "皮具": "Đồ da", "制衣": "May mặc", "食品": "Thực phẩm"}
btc = pd.DataFrame({"tu_chung_zh": list(TU_CHUNG), "dich_goi_y": list(TU_CHUNG.values()), "anh_duyet": "", "anh_ghi_chu": ""})
btc = file_duyet("P7_tu_chung_ten_hang_v2_editted.csv", btc, ["tu_chung_zh"])   # v2: bảng mở rộng 02/10 (v1 chỉ 16 từ)
tc_map = {r.tu_chung_zh: (r.anh_duyet.strip() if r.anh_duyet.strip() and r.anh_duyet.strip().upper() != "OK" else r.dich_goi_y) for r in btc.itertuples()}
LATIN_KHAC = {"other", "others", "ohter", "other/其他", "other/其它", "其他/other", "其他/others"}
# Chú thích loại hàng trong ngoặc sau tên hãng: dịch nghĩa, không phiên âm (天王（食品）→ 'Tianwang (thực phẩm)')
LOAI_NGOAC = {"食品": "thực phẩm", "食物": "thực phẩm", "食品类": "thực phẩm", "母婴": "mẹ và bé", "美妆": "mỹ phẩm", "彩妆": "mỹ phẩm",
              "数码": "kỹ thuật số", "鞋": "giày", "箱包": "túi xách", "洗护": "chăm sóc", "收纳整理": "đồ lưu trữ", "服饰": "thời trang",
              "奶嘴": "núm vú", "印度尼西亚": "Indonesia", "布料床单": "vải ga giường", "钓鱼类玩具": "đồ chơi câu cá", "床上用品": "đồ dùng giường ngủ",
              "车载用品": "đồ dùng ô tô", "汽配": "phụ tùng ô tô", "服装鞋帽": "quần áo giày mũ", "清洁工具": "dụng cụ vệ sinh", "电器": "đồ điện"}
CONG_TY = re.compile(r"有限公司|公司|工厂|厂$|厂（|供应链|工艺品$|纺织品|电子产品$")


def phien_am(s):
    """Phiên âm phần chữ Hán, giữ nguyên chữ số và ký tự khác (70迈 → 70mai)."""
    out = "".join("".join(lazy_pinyin(t)) if CJK.match(t) else t for t in re.findall(r"[一-鿿]+|[^一-鿿]+", s))
    return out[:1].upper() + out[1:]


def ten_hang(z):
    zs = z.strip()
    if has_cjk(zs):   # bỏ ký hiệu thừa ở đầu/cuối ('-江源', '沐鸣-'), dấu câu Trung → ASCII ('简，澈' → '简, 澈')
        zs = re.sub(r"\s*，\s*", ", ", zs).replace("、", ", ").strip(" -_.·/\\|~*")
    if zs in tc_map:
        return tc_map[zs], "tu_chung"
    if zs.lower() in LATIN_KHAC:
        return "Khác", "tu_chung"
    if not has_cjk(zs):
        return zs, "latin"          # tên hãng Latin: chép nguyên bản Trung (máy từng dịch 'zx' → 'từ đó', 'LIFE HOME' → 'NHÀ CUỘC SỐNG')
    m = re.match(r"^(.*?)[（(]([^）)]*)[）)]?\s*$", zs)
    chinh, ngoac = (m.group(1).strip(), m.group(2).strip()) if m else (zs, "")
    if len(CJK.findall(chinh)) > 7 or CONG_TY.search(chinh):
        return None, "dai"          # mô tả hàng / tên công ty nằm trong ô 品牌
    latin = re.findall(r"[A-Za-z][A-Za-z0-9&'\-]+(?:\s+[A-Za-z0-9&'\-]+)*", chinh)
    hau_to = next((h for h in HAU_TO_LOAI if chinh.endswith(h) and 1 <= len(chinh) - len(h) <= 4 and has_cjk(chinh[:-len(h)])), None)
    if latin:
        goc, loai = " ".join(latin).strip(), "giu_latin"
    elif hau_to:
        goc, loai = f"{HAU_TO_LOAI[hau_to]} {phien_am(chinh[:-len(hau_to)])}", "pinyin_loai"
    else:
        goc, loai = phien_am(chinh), "pinyin"
    if ngoac:
        if ngoac in LOAI_NGOAC:
            goc = f"{goc} ({LOAI_NGOAC[ngoac]})"
        elif not has_cjk(ngoac):
            goc = f"{goc} ({ngoac})"
        else:
            return None, "dai"      # chú thích chữ Hán lạ: không đoán
    return goc, loai


mb = V.name_zh.eq("品牌")
kq = [ten_hang(z) for z in V.loc[mb, "v_zh"]]
truoc = V.v_vi.copy()
idx_b = V.index[mb]
dai = []
for i, (moi, loai) in zip(idx_b, kq):
    if loai == "dai":
        dai.append(V.at[i, "pair_id"])
    elif moi is not None:
        V.at[i, "v_vi"] = moi
them_ly_do(dai, "DS09_dai")
n75 = ghi_log("P7_ten_hang", "v_vi", V, truoc, V.v_vi)
bh = V.loc[idx_b].assign(truoc=truoc.loc[idx_b], sau=V.loc[idx_b, "v_vi"], loai=[l for _, l in kq])
bh = bh[(bh.truoc != bh.sau) & bh.loai.isin(["pinyin", "giu_latin", "pinyin_loai"])].drop_duplicates(["v_zh", "sau"])
mau_kiem("P7_ten_hang", bh, 200, ["v_zh", "truoc", "sau", "loai"], cho_phep_sai=10, viec="tên hãng chuyển pinyin (anh đọc 200 mẫu, sai ≤ 10)",
         phien="v5")   # v5: v2 ~5%, v3 ~2,5%, v4 1% từ chung/mô tả → bổ sung + quét toàn bộ đuôi loại hàng (02/10)
cong("P7_tu_chung_ten_hang", "danh sách từ chung trong 品牌", "dat" if (btc.anh_duyet.str.strip() != "").all() else "cho_duyet",
     f"{len(btc)} từ chung, còn {(btc.anh_duyet.str.strip() == '').sum()} chưa duyệt")

# 7.6 Tiếng Anh/pinyin theo bảng title (THAY, HANVIET)
# TẮT theo quyết định 02/10: phần xử lý tiếng Anh do Quang làm sau. Đổi XU_LY_TIENG_ANH = True ở P0 để bật lại.
n76 = 0
if XU_LY_TIENG_ANH:
    tt = pd.read_csv(TITLE_AUDIT / "title_latin_tokens_editted.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
    tt = tt[tt.hanh_dong.isin(["THAY", "HANVIET"]) & (tt.thay_bang.str.strip() != "")]
    thay = {t.lower(): r for t, r in zip(tt.token, tt.thay_bang.str.strip())}
    TOKEN = re.compile(r"[A-Za-z][A-Za-z\-']+")
    truoc = V.v_vi.copy()
    cand = V.v_vi.map(lambda s: any(t.lower() in thay for t in TOKEN.findall(s)))
    for i in V.index[cand & ~la_ma & (V.name_zh != "品牌")]:
        v, z = V.at[i, "v_vi"], V.at[i, "v_zh"].lower()
        for t in set(TOKEN.findall(v)):
            if t.lower() in thay and t.lower() not in z:
                v = re.sub(rf"(?<![A-Za-zÀ-ỹ]){re.escape(t)}(?![A-Za-zÀ-ỹ])", thay[t.lower()], v)
        V.at[i, "v_vi"] = v
    n76 = ghi_log("P7_tieng_anh", "v_vi", V, truoc, V.v_vi)
    cong("P7_tieng_anh", "bảng từ tiếng Anh của title", "cho_duyet" if (pd.read_csv(TITLE_AUDIT / "title_latin_tokens_editted.csv", dtype=str, keep_default_na=False,
         encoding="utf-8-sig").anh_duyet.str.strip() == "").all() else "dat", "dùng bảng title_latin_tokens_editted.csv (anh đang duyệt cho title)")
else:
    cong("P7_tieng_anh", "bảng từ tiếng Anh của title", "khong_can", "TẮT: phần tiếng Anh do Quang xử lý sau (XU_LY_TIENG_ANH = False)")

# 7.7 Thuật ngữ chung với title
tc = pd.read_csv(TITLE_AUDIT / "title_term_consistency.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
col = [c for c in tc.columns if c.startswith("cach_dich_vi")][0]
CAT_TN = {"密码箱": {"bags"}, "拉杆箱": {"bags"}}
CHUAN_TN = {"行车记录仪": "camera hành trình", "充电宝": "sạc dự phòng", "密码箱": "vali khóa số", "拉杆箱": "vali kéo", "面膜": "mặt nạ", "双肩包": "ba lô",
            "爆款": "bán chạy", "网红": "nổi tiếng trên mạng",
            # 02/10: Nhật Anh giao Claude chọn theo ngữ cảnh (đã tra cách dùng của người bán/trang mỹ phẩm Việt)
            "一件代发": "dropship", "补水": "cấp nước", "老爹鞋": "giày bố"}
GHI_CHU_TN = {"补水": "Claude 02/10: 补水 = cấp nước (bổ sung nước cho da), khác 保湿 = dưỡng ẩm (giữ nước); chỉ đổi 'dưỡng ẩm' khi bản Trung không có 保湿/润; sửa 'uống nước', 'hấp thụ nước', 'hydrat hóa', 'dưỡng ẩm và dưỡng ẩm'",
              "一件代发": "Claude 02/10: 一件代发 = nhà cung cấp gửi thẳng từng đơn (dropship); chỉ thay các cụm dịch sai 'vận chuyển thả', 'giao hàng một món/chiếc/mảnh', 'giao hàng tận nơi' (trừ khi bản Trung có 上门); không thay 'giao hàng' đứng riêng",
              "老爹鞋": "Claude 02/10: chọn 'giày bố' (người bán Việt gọi 'dad shoes (giày bố)'; tiếng Anh để Quang xử lý); sửa 'giày bố già', 'giày ông già'",
              "爆款": "Claude 02/10: biến thể 'bùng nổ', 'kiểu nổ', 'mẫu nổ' là dịch sai; 'hot' (ASCII) giữ nguyên",
              "网红": "Claude 02/10: 'người nổi tiếng' sai ngữ cảnh (网红款 = kiểu đang nổi trên mạng); 'internet', 'net red' (ASCII) không bị thay",
              }
btn = pd.DataFrame({"thuat_ngu_zh": tc.thuat_ngu_zh, "cac_cach_dich": tc[col], "chuan": tc.thuat_ngu_zh.map(CHUAN_TN).fillna(""),
                    "ghi_chu": tc.thuat_ngu_zh.map(GHI_CHU_TN).fillna(""), "anh_duyet": "", "anh_ghi_chu": ""})
btn = file_duyet("P7_thuat_ngu_editted.csv", btn, ["thuat_ngu_zh"])
# File duyệt tạo trước ngày 02/10 chưa có chuẩn của 爆款/网红: cột 'chuan' và 'ghi_chu' là gợi ý của Claude, luôn lấy theo code
# (cột của anh không bị đụng).
btn["chuan"] = btn.thuat_ngu_zh.map(CHUAN_TN).fillna("")
btn["ghi_chu"] = btn.thuat_ngu_zh.map(GHI_CHU_TN).fillna("")
truoc = V.v_vi.copy()
# Cụm thay riêng, xét trước cụm ngắn, để không lặp ý ('người nổi tiếng trên Internet' → 'nổi tiếng trên mạng', không thành '... trên mạng trên Internet')
CUM_RIENG = {"网红": {"người nổi tiếng trên internet": "nổi tiếng trên mạng", "người nổi tiếng trên mạng": "nổi tiếng trên mạng"},
             "爆款": {"mô hình bán hàng bùng nổ": "mẫu bán chạy", "bán hàng bùng nổ": "bán chạy", "mô hình bùng nổ": "mẫu bán chạy",
                      "mẫu nổ": "mẫu bán chạy", "kiểu nổ": "kiểu bán chạy"},
             "一件代发": {"vận chuyển thả": "dropship", "thả vận chuyển": "dropship", "giao hàng một món": "dropship",
                      "giao hàng một chiếc": "dropship", "giao hàng một mảnh": "dropship", "giao một chiếc": "dropship",
                      "giao hàng tận nơi": "dropship"},
             "补水": {"dưỡng ẩm và dưỡng ẩm": "cấp nước và dưỡng ẩm", "uống nước": "cấp nước", "hấp thụ nước": "cấp nước",
                    "hydrat hóa": "cấp nước", "dưỡng ẩm": "cấp nước"},
             "老爹鞋": {"giày bố già": "giày bố", "giày ông già": "giày bố"}}
# Điều kiện theo bản Trung cho một số cụm (để không sửa nhầm câu đúng)
DIEU_KIEN_CUM = {("一件代发", "giao hàng tận nơi"): lambda z: "上门" not in z,
                 ("补水", "dưỡng ẩm"): lambda z: not re.search("保湿|保濕|润", z)}
# Thuật ngữ chỉ thay theo CUM_RIENG, không thay các biến thể lấy tự động từ title (vd 'giao hàng' là cụm thông dụng)
CHI_BIEN = {"一件代发": [], "补水": ["bổ sung nước"], "老爹鞋": []}


def thay_giu_hoa(cu_, moi_, v):
    """Thay không phân biệt hoa thường, giữ kiểu chữ hoa của cụm gốc:
    viết hoa mọi chữ ('Dưỡng Ẩm' → 'Cấp Nước'), hoa chữ đầu ('[Mô hình...' → '[Mẫu ...'), còn lại giữ nguyên."""
    def kieu(m):
        g = m.group(0)
        tu = [w for w in g.split() if w[:1].isalpha()]
        if len(tu) > 1 and all(w[:1].isupper() for w in tu):
            return " ".join(w[:1].upper() + w[1:] for w in moi_.split())
        return moi_[:1].upper() + moi_[1:] if g[:1].isupper() else moi_
    return re.sub(re.escape(cu_), kieu, v, flags=re.I)


for r in btn.itertuples():
    chuan = r.anh_duyet.strip() if r.anh_duyet.strip() and r.anh_duyet.strip().upper() != "OK" else r.chuan.strip()
    if not chuan:
        continue
    if r.thuat_ngu_zh in CUM_RIENG and not (r.anh_duyet.strip() and r.anh_duyet.strip().upper() != "OK"):
        mz = V.v_zh.str.contains(r.thuat_ngu_zh, regex=False)
        for i in V.index[mz]:
            v0 = v = V.at[i, "v_vi"]
            z = V.at[i, "v_zh"]
            for cu_, moi_ in sorted(CUM_RIENG[r.thuat_ngu_zh].items(), key=lambda x: -len(x[0])):
                dk = DIEU_KIEN_CUM.get((r.thuat_ngu_zh, cu_))
                if dk is None or dk(z):
                    v = thay_giu_hoa(cu_, moi_, v)
            if v != v0 and v0[:1].isupper():
                v = v[:1].upper() + v[1:]
            V.at[i, "v_vi"] = v
    # chỉ thay các cách dịch tiếng Việt; bỏ qua biến thể thuần ASCII (vd 'mask') để không tạo câu lai trong cụm tiếng Anh
    bien = sorted({p.rsplit(":", 1)[0].strip() for p in r.cac_cach_dich.split(";") if p.strip()} - {chuan}, key=len, reverse=True)
    bien = [b for b in bien if not b.isascii()]
    if r.thuat_ngu_zh in CHI_BIEN:
        bien = CHI_BIEN[r.thuat_ngu_zh]
    mm = V.v_zh.str.contains(r.thuat_ngu_zh, regex=False)
    if r.thuat_ngu_zh in CAT_TN:   # thuật ngữ chỉ đúng trong một số category (密码箱 ở mẹ và bé có thể là đồ chơi)
        mm &= V.row.map(df.category).isin(CAT_TN[r.thuat_ngu_zh])
    for i in V.index[mm]:
        v0 = v = V.at[i, "v_vi"]
        for b in bien:
            if b and chuan.lower() not in v.lower():
                v = thay_giu_hoa(b, chuan, v)
        if v != v0 and v0[:1].isupper():
            v = v[:1].upper() + v[1:]
        V.at[i, "v_vi"] = v
n77 = ghi_log("P7_thuat_ngu", "v_vi", V, truoc, V.v_vi)
cong("P7_thuat_ngu", "bảng thuật ngữ chung title–description", "dat" if (btn.anh_duyet.str.strip() != "").all() else "cho_duyet",
     f"{len(btn)} thuật ngữ, {(btn.chuan.str.strip() != '').sum()} có chuẩn tạm")
luu("p7")
ket_luan(f"Từ điển giá trị: đổi {so(n71)} giá trị ({len(TD7)} mục, {len(ds7)} mục khác cách dịch đa số); ngữ cảnh: {so(n72)}.",
         f"DS17 (lỗi trong chuỗi dài): {so(n17)} cặp gắn mã; địa danh: dựng lại {so(n74)} giá trị, {so(n18)} cặp gắn DS18.",
         f"Tên hãng: đổi {so(n75)} giá trị; tiếng Anh theo bảng title: {so(n76)}; thuật ngữ chung: {so(n77)}.")

## P8. Số và đơn vị (DS10, DS03, DS20)

1. **Giá trị là mã thật:** bản Việt chép nguyên bản Trung. Mã thật nghĩa là thuộc vai trò mã/giấy tờ VÀ giá trị Trung không có chữ Hán (SC1043..., XRNBS), hoặc là số giấy phép/đăng ký: tên chứa 许可证, 备案, 批准文号, 注册证, 标准号, 证号, 准字 và giá trị có từ 4 chữ số liền nhau (粤妆20190234, 国妆特字G20191510). Giá trị chữ mô tả nằm dưới 型号/货号 thì vẫn dịch như thường. Mã còn chữ Hán thì thêm chú thích tỉnh/cấp quốc gia sau mã (chốt 03/10): "粤妆20170110 (Quảng Đông)", "国妆网备进字（沪）2024007457 (quốc gia, Thượng Hải)"; ngày "2023年05月24日" → "24/05/2023".
2. **Số thập phân bị tách** ("53, 5"): ghép lại thành "53.5" khi bản Trung có "53.5".
3. **Năm bị đổi:** khi mỗi bên chỉ có một năm 20xx mà khác nhau, lấy năm bên Trung.
4. **Danh sách lệch thứ tự:** sắp lại bản Việt theo số của bản Trung khi đó là một hoán vị. Không sắp được thì gắn `DS03`.
5. **Quy đổi 斤 → kg (luật kiểm bản 2, 06/10, `src/doi_chieu.py`):** 1 斤 = 0,5 kg. Bản Việt viết kg mà giữ nguyên số của bản Trung ("190-215斤" → "190-215 kg") thì chia đôi các số đó (95-107.5 kg). Giá tính theo 斤 ("30元/斤") mà bản Việt viết theo kg, hoặc số không bằng số gốc cũng không bằng một nửa, thì gắn `DS10`. "公斤" (= kg) và "千斤顶" (cái kích xe) không bị đụng. Luật số cũ (mục 6) không bắt được lỗi này vì coi là "không lệch" ngay khi số bản Việt giống hệt bản Trung.
6. **Số còn lệch sau khi quy đổi:** chữ số Hán đơn, 万, 纯 → 100% không tính là lệch (斤 → kg đã kiểm ở mục 5). Còn lệch thì gắn `DS10`.
7. **Cảnh báo đơn vị (chỉ gắn nhãn `CB_*`, không sửa, không loại cặp):** cùng kiểu hở với 斤 (số giữ nguyên, đơn vị đổi) mà luật số cũ không thấy. `CB_cun`: N寸 mà bản Việt viết cùng số theo cm/mm/m hoặc dịch "tấc/thốn" (寸 ở đây chủ yếu là inch). `CB_luong`: N两 (= 50 N gram) mà bản Việt viết N gram hoặc dùng "lạng/lượng/tael" (lạng của Việt = 100 g). `CB_van`: N万 (= N × 10.000) mà bản Việt giữ nguyên số N. `CB_ty`: N丝 (độ dày 0,01 mm của túi nhựa) mà bản Việt dịch "lụa/tơ". Danh sách cho người xem: `data/processed/desc/p8_canh_bao_don_vi.csv`.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "src"))
from doi_chieu import canh_bao_don_vi, kiem_jin   # luật kiểm đơn vị, bản 2 (06/10)

truoc = V.v_vi.copy()
la_ma = V.la_ma
V.loc[la_ma, "v_vi"] = V.loc[la_ma, "v_zh"]
them_nhan(V.loc[la_ma, "pair_id"], "chep_ma")

# 03/10 (Nhật Anh chốt): mã còn chữ Hán thì thêm chú thích tiếng Việt sau mã: 粤妆20170110 → "粤妆20170110 (Quảng Đông)"
TINH = {"粤": "Quảng Đông", "粵": "Quảng Đông", "沪": "Thượng Hải", "浙": "Chiết Giang", "鲁": "Sơn Đông", "陕": "Thiểm Tây",
        "皖": "An Huy", "云": "Vân Nam", "滇": "Vân Nam", "苏": "Giang Tô", "津": "Thiên Tân", "黑": "Hắc Long Giang",
        "赣": "Giang Tây", "湘": "Hồ Nam", "桂": "Quảng Tây", "冀": "Hà Bắc", "鄂": "Hồ Bắc", "吉": "Cát Lâm", "川": "Tứ Xuyên",
        "蜀": "Tứ Xuyên", "豫": "Hà Nam", "京": "Bắc Kinh", "闽": "Phúc Kiến", "辽": "Liêu Ninh", "晋": "Sơn Tây",
        "渝": "Trùng Khánh", "琼": "Hải Nam", "贵": "Quý Châu", "黔": "Quý Châu", "甘": "Cam Túc", "陇": "Cam Túc",
        "宁": "Ninh Hạ", "新": "Tân Cương", "蒙": "Nội Mông", "青": "Thanh Hải", "藏": "Tây Tạng"}
NGAY = re.compile(r"(\d{4})年(\d{1,2})月(\d{1,2})日")


def chu_thich_ma(v):
    if not has_cjk(v):
        return v
    m = NGAY.fullmatch(v.strip())
    if m:   # ngày tháng lọt vào nhóm mã: đổi sang kiểu Việt
        return f"{int(m.group(3)):02d}/{int(m.group(2)):02d}/{m.group(1)}"
    han = CJK.findall(v)
    phan = (["quốc gia"] if han and han[0] == "国" else []) + list(dict.fromkeys(TINH[c] for c in han if c in TINH))
    return f"{v} ({', '.join(phan)})" if phan else v


V.loc[la_ma, "v_vi"] = V.loc[la_ma, "v_vi"].map(chu_thich_ma)
ma_han = V[la_ma & V.v_vi.map(has_cjk)]
print(f"Mã còn chữ Hán: {so(len(ma_han))} giá trị; có chú thích: {so(ma_han.v_vi.str.endswith(')').sum())}; "
      f"không chú thích được: {so((~ma_han.v_vi.str.endswith(')')).sum())}")


def ghep_thap_phan(z, v):
    for a, b in re.findall(r"(\d+)\.(\d+)", z):
        v = v.replace(f"{a}, {b}", f"{a}.{b}")
    return v


YEAR = re.compile(r"20\d\d")


def sua_nam(z, v):
    yz, yv = set(YEAR.findall(z)), set(YEAR.findall(v))
    if len(yz) == 1 and len(yv) == 1 and yz != yv:
        return v.replace(yv.pop(), yz.pop())
    return v


V["v_vi"] = [sua_nam(z, ghep_thap_phan(z, v)) for z, v in zip(V.v_zh, V.v_vi)]
n81 = ghi_log("P8", "v_vi", V, truoc, V.v_vi)

# 06/10: quy đổi 斤 → kg. Luật số cũ (lech_so, bên dưới) bỏ qua khi số bản Việt giống hệt bản Trung, nên "190-215斤" → "190-215 kg"
# lọt; ở đây kiểm đúng giá trị: bản Việt giữ nguyên số thì chia đôi; giá theo 斤 hoặc số lệch khác thì gắn DS10.
truoc = V.v_vi.copy()
kq_jin = [kiem_jin(z, v) if "斤" in z else ("khong_ap_dung", v) for z, v in zip(V.v_zh, V.v_vi)]
V["v_vi"] = [k[1] for k in kq_jin]
tt_jin = pd.Series([k[0] for k in kq_jin], index=V.index)
n_jin = ghi_log("P8_jin", "v_vi", V, truoc, V.v_vi)
jin_loi = V.loc[tt_jin.isin(["gia_theo_can", "lech_khac"]) & ~la_ma, "pair_id"].unique()
print("斤 → kg:", dict(Counter(tt_jin[tt_jin != "khong_ap_dung"])), f"| sửa {so(n_jin)} giá trị, gắn DS10 {so(len(jin_loi))} cặp")

# 06/10: cảnh báo đơn vị 寸, 两, 万, 丝 (chỉ gắn nhãn CB_*, không sửa, không loại cặp): số bản Việt giữ nguyên nhưng đơn vị
# đổi, luật số cũ không thấy được. Danh sách cho người xem: data/processed/desc/p8_canh_bao_don_vi.csv
co_dv = V.v_zh.str.contains("[寸两万丝]", regex=True) & ~la_ma
cb_ds = []
for pid, z, v in zip(V.pair_id[co_dv], V.v_zh[co_dv], V.v_vi[co_dv]):
    for ma in canh_bao_don_vi(z, v):
        cb_ds.append({"pair_id": pid, "ma": ma, "gia_tri_zh": z, "gia_tri_vi": v})
cb = pd.DataFrame(cb_ds, columns=["pair_id", "ma", "gia_tri_zh", "gia_tri_vi"])
for ma, g in cb.groupby("ma"):
    them_nhan(g.pair_id, ma)
cb.to_csv(OUT / "p8_canh_bao_don_vi.csv", index=False, encoding="utf-8-sig")
print("Cảnh báo đơn vị:", cb.ma.value_counts().to_dict(), "→ data/processed/desc/p8_canh_bao_don_vi.csv")

# danh sách lệch thứ tự (chỉ xét cặp có ≥ 3 giá trị)
numseq = lambda s: tuple(DIG.findall(s))
dem = V.pair_id.value_counts()
xet = V[~la_ma & V.pair_id.isin(dem.index[dem >= 3])]
truoc = V.v_vi.copy()
n_sap, ds03 = 0, []
for p, g in xet.groupby("pair_id"):
    nz = [numseq(x) for x in g.v_zh]
    if sum(bool(x) for x in nz) < 3 or len(set(nz)) < len(nz):
        continue
    nv = [numseq(x) for x in g.v_vi]
    if nz == nv:
        continue
    jin = ["斤" in a and "kg" in b.lower() for a, b in zip(g.v_zh, g.v_vi)]   # 斤 → kg là quy đổi đúng
    if all(j or set(a) <= set(b) for a, b, j in zip(nz, nv, jin)):
        continue
    if sorted(nz) == sorted(nv):
        thu_tu = [nv.index(x) for x in nz]
        moi = [g.v_vi.iloc[j] for j in thu_tu]
        V.loc[g.index, "v_vi"] = moi
        n_sap += 1
    else:
        ds03.append(p)
them_ly_do(ds03, "DS03")
ghi_log("P8_sap_lai", "v_vi", V, truoc, V.v_vi)

# số còn lệch (cấp cặp)
NUM = re.compile(r"\d+(?:\.\d+)?")


def nums(s):
    s = re.sub(r"(?<=\d)\.(?=\d{3}(?!\d))", "", s)
    s = re.sub(r"(?<=\d),(?=\d)", ".", s)
    return sorted(float(x) for x in NUM.findall(s))


def lech_so(z, v):
    sz, sv = sorted(float(x) for x in NUM.findall(z)), nums(v)
    if sz == sv:
        return False
    ok = Counter(sz)
    if "斤" in z and "kg" in v.lower():
        ok += Counter(x / 2 for x in sz)
    if "万" in z:
        ok += Counter(x / 100 for x in sz) + Counter(x * 10000 for x in sz)
    if "纯" in z:
        ok += Counter([100.0])
    cn = {float(CN_NUM[c]) for c in z if c in CN_NUM}
    thua = Counter(sv) - ok
    thieu = Counter(sz) - Counter(sv)
    thieu = Counter({x: c for x, c in thieu.items() if not (x / 2 in sv or x / 100 in sv or x * 10000 in sv)})
    if not thieu and all(x in cn for x in thua):
        return False
    return bool(thua or thieu)


ghep = V[~la_ma].groupby("pair_id").agg(z=("v_zh", " | ".join), v=("v_vi", " | ".join))
co_so = ghep.z.map(lambda s: bool(DIG.search(s))) | ghep.v.map(lambda s: bool(DIG.search(s)))
lech = [p for p, z, v in zip(ghep.index[co_so], ghep.z[co_so], ghep.v[co_so]) if lech_so(z, v)]
n10 = them_ly_do(lech, "DS10")
n10 += them_ly_do(jin_loi, "DS10")
luu("p8")
ket_luan(f"Chép nguyên {so(la_ma.sum())} giá trị mã/giấy tờ; sửa thập phân bị tách và năm: {so(n81)} giá trị (gồm phần chép mã).",
         f"Sắp lại {so(n_sap)} danh sách; {so(len(ds03))} cặp DS03; {so(n10)} cặp còn lệch số (DS10).",
         f"Quy đổi 斤 → kg: sửa {so(n_jin)} giá trị; {so(len(jin_loi))} cặp gắn DS10 (giá theo 斤 hoặc số lệch khác).",
         f"Cảnh báo đơn vị (chỉ gắn nhãn, không loại): {cb.ma.value_counts().to_dict()}.")

## P9. Định dạng (DS22, DS23, DS14)

- **Dấu chấm cuối do máy thêm:** bỏ đi, trừ khi bản Trung cũng kết thúc bằng dấu chấm, hoặc giá trị kết thúc bằng "...".
- **Chữ tiếng Việt IN HOA toàn bộ:** đưa về viết hoa chữ đầu, ví dụ "KHÔNG" → "Không".
- **Số thập phân:** đổi mọi "a,b" thành "a.b" khi bản Trung có đúng "a.b" và chỉ có 1–2 chữ số sau dấu (02/10 bỏ điều kiện có đơn vị). Ví dụ "7,5 * 7,5 * 2,5CM" → "7.5 * 7.5 * 2.5CM", "F1,55" → "F1.55". Số hàng nghìn "1.000" và danh sách "9,12,24" không bị đụng.

In [ ]:
def in_hoa_viet(s):
    chu = [c for c in s if c.isalpha() and not CJK.match(c)]
    return len(chu) >= 4 and all(c.isupper() for c in chu) and any(not c.isascii() for c in chu)


CO_DON_VI = re.compile(r"\d\s?(?:cm|mm|m|mét|kg|g|ml|l|lít|inch|w|v|mah|cbm|%)(?![A-Za-zÀ-ỹ])", re.I)


def dinh_dang(z, v):
    if v.endswith(".") and not v.endswith("...") and not z.rstrip().endswith(("。", ".")) and not re.search(r"(?:\b[A-Za-z]\.){2,}$", v):
        v = v.rstrip(".").rstrip()
    if in_hoa_viet(v):
        v = v[:1].upper() + v[1:].lower()
    # Dấu thập phân: đổi mọi 'a,b' thành 'a.b' khi bản Trung có đúng 'a.b' (1–2 chữ số sau dấu).
    # 02/10: bỏ điều kiện "có đơn vị" (Nhật Anh chốt) — điều kiện bản Trung có đúng a.b đã đủ để không đụng danh sách '9,12,24'.
    for a, b in set(re.findall(r"(\d+)\.(\d{1,2})(?!\d)", z)):
        v = re.sub(rf"(?<![\d.,]){a},{b}(?!\d)", f"{a}.{b}", v)
    return v


truoc = V.v_vi.copy()
V["v_vi"] = [dinh_dang(z, v) for z, v in zip(V.v_zh, V.v_vi)]
n9 = ghi_log("P9", "v_vi", V, truoc, V.v_vi)
luu("p9")
ket_luan(f"Định dạng lại {so(n9)} giá trị (dấu chấm cuối, chữ in hoa, dấu thập phân).")

## P10. Kiểm nghĩa phần đuôi dài (DS04, đuôi DS15/17)

"Đuôi dài" là các giá trị chữ Hán không có trong từ điển P7, không phải mã, không phải tên hãng, và cặp chưa bị loại.

**Tầng 1** (`duyet/P10_tang1_v7_editted.csv`, chỉ tạo sau khi Claude xong P7):
- Gồm khoảng 3.000 cặp (zh, vi) phổ biến nhất, cộng các cặp có tỉ lệ độ dài bất thường (DS04).
- Claude điền cột `claude_nhan` = `DUNG` / `SAI` / `KHONG_CHAC`, và `claude_sua` nếu sửa được.
- Cặp `SAI` có bản sửa thì thay bản dịch; `SAI` không có bản sửa thì gắn mã `P10_SAI`.
- **Cổng:** anh kiểm 30 mẫu `SAI` và 30 mẫu `DUNG`.

**Tầng 2** (phần còn lại, ~130 nghìn cặp hiếm):
- **Phương án A (chốt 03/10, `DUNG_QWEN = False` ở P0):** cặp có giá trị tầng 2 gắn mã `T2_CHUA_KIEM`, không vào train nhưng vẫn giữ trong file.
  - Lý do: Claude chấm 300 cặp ngẫu nhiên thấy ~26% sai nghĩa; Qwen2.5-7B (Colab, 4-bit) chạy thử trên 100 cặp hiệu chỉnh chỉ đạt precision 28–32% ở mọi ngưỡng, gần bằng chọn bừa, nên không dùng được.
  - Vẫn xuất `P10_tang2_cho_qwen.csv` để chấm lại sau nếu có cách tốt hơn (vd LLM mạnh hơn).
- Khi `DUNG_QWEN = True`: đọc `P10_tang2_ket_qua.csv` và `P10_qwen_hieu_chinh.csv`; cặp `SAI_NGHIA` chỉ bị loại khi precision ≥ 80% (và ≥ 20 cờ) trên 200 cặp đợt 2.

Cặp tầng 1 chưa được kiểm (hoặc Claude gắn `KHONG_CHAC`) thì gắn nhãn `chua_kiem_nghia`: vẫn vào train, không bao giờ dùng làm test.

In [ ]:
loai = P.set_index("pair_id").ly_do.ne("")
vai = P.set_index("pair_id").vai_tro
la_ma = V.la_ma
duoi = V[V.v_zh.map(has_cjk) & ~V.v_zh.isin(gt_map.keys()) & ~V.pair_id.map(loai) & ~la_ma & V.name_zh.ne("品牌")
         & ~V.dd_dung_lai].copy()
uniq = (duoi.groupby(["v_zh", "v_vi"]).agg(so_lan=("pair_id", "size"), ten_thuoc_tinh=("name_zh", "first"), pair_vd=("pair_id", "first"))
        .reset_index().sort_values("so_lan", ascending=False))
n_cjk = uniq.v_zh.map(lambda s: len(CJK.findall(s)))
n_lat = uniq.v_zh.map(lambda s: len(re.findall(r"[A-Za-z]", s)))
n_chu = uniq.v_vi.map(lambda s: sum(c.isalpha() for c in s))
uniq["ti_le"] = ((n_chu - n_lat).clip(lower=0) / n_cjk).round(2)
q1, q99 = uniq.ti_le[n_cjk >= 4].quantile([.01, .99])
uniq["bat_thuong"] = (n_cjk >= 4) & ((uniq.ti_le < q1) | (uniq.ti_le > q99))
tang1 = pd.concat([uniq.head(3000), uniq.iloc[3000:][uniq.iloc[3000:].bat_thuong]])
tang2 = uniq.drop(tang1.index)
t1 = pd.DataFrame({"gia_tri_zh": tang1.v_zh, "gia_tri_vi": tang1.v_vi, "ten_thuoc_tinh": tang1.ten_thuoc_tinh, "so_lan": tang1.so_lan,
                   "ti_le_do_dai": tang1.ti_le, "do_dai_bat_thuong": tang1.bat_thuong.astype(int),
                   "product_id_vd": P.set_index("pair_id").loc[tang1.pair_vd, "product_id"].values,
                   "claude_nhan": "", "claude_sua": "", "ghi_chu": ""})
# File tầng 1 chỉ tạo khi P4–P7 đã chốt phía Claude: tạo sớm thì khoá (zh, vi) sẽ lệch khi từ điển còn đổi.
if CLAUDE_XONG["P7"]:
    T1 = file_duyet("P10_tang1_v7_editted.csv", t1, ["gia_tri_zh", "gia_tri_vi"])
else:
    T1 = t1.iloc[:0].astype(str)
    print(f"Chưa tạo file tầng 1 ({so(len(t1))} cặp dự kiến): chờ Claude xong P7")
tang2[["v_zh", "v_vi", "ten_thuoc_tinh", "so_lan"]].rename(columns={"v_zh": "gia_tri_zh", "v_vi": "gia_tri_vi"}).to_csv(
    DUYET / "P10_tang2_cho_qwen.csv", index=False, encoding="utf-8-sig")
print(f"Đuôi dài: {so(len(duoi))} giá trị / {so(len(uniq))} cặp khác nhau; tầng 1: {so(len(tang1))}; tầng 2: {so(len(tang2))} (→ P10_tang2_cho_qwen.csv)")

# Áp tầng 1
key = list(zip(duoi.v_zh, duoi.v_vi))
nhan1 = {(r.gia_tri_zh, r.gia_tri_vi): (r.claude_nhan.strip().upper(), r.claude_sua.strip()) for r in T1.itertuples()}
truoc = V.v_vi.copy()
sai_drop, kc = [], []
for i, k in zip(duoi.index, key):
    nh, sua = nhan1.get(k, ("", ""))
    if nh == "SAI" and sua:
        V.at[i, "v_vi"] = sua
    elif nh == "SAI":
        sai_drop.append(V.at[i, "pair_id"])
    elif nh == "KHONG_CHAC":
        kc.append(V.at[i, "pair_id"])
n10s = ghi_log("P10_tang1", "v_vi", V, truoc, V.v_vi)
them_ly_do(sai_drop, "P10_SAI")
co_nhan = T1[T1.claude_nhan.str.strip() != ""]
if not CLAUDE_XONG["P10_tang1"]:
    cong("P10_tang1", "Claude đọc tầng 1", "cho_claude",
         f"{len(co_nhan)}/{len(T1)} cặp đã có claude_nhan ({rel(DUYET / 'P10_tang1_v7_editted.csv')})" if len(T1)
         else f"chờ Claude xong P7 rồi mới tạo file tầng 1 ({so(len(t1))} cặp dự kiến)")
else:
    mau_kiem("P10_tang1_SAI", co_nhan[co_nhan.claude_nhan.str.upper() == "SAI"], 30, ["gia_tri_zh", "gia_tri_vi", "claude_nhan", "claude_sua"], viec="Claude gắn SAI")
    mau_kiem("P10_tang1_DUNG", co_nhan[co_nhan.claude_nhan.str.upper() == "DUNG"], 30, ["gia_tri_zh", "gia_tri_vi", "claude_nhan"], viec="Claude gắn DUNG")

# Áp tầng 2
kq2, hc = DUYET / "P10_tang2_ket_qua.csv", DUYET / "P10_qwen_hieu_chinh.csv"
n_q = 0
if not DUNG_QWEN:
    # Phương án A (Nhật Anh chốt 03/10): Qwen2.5-7B không phân biệt được (precision 28–32% ≈ tỉ lệ sai nền 25%)
    # → cặp có giá trị tầng 2 gắn mã T2_CHUA_KIEM: không vào train, vẫn giữ trong file để chấm lại sau nếu có cách tốt hơn.
    k_t2 = set(zip(tang2.v_zh, tang2.v_vi))
    drop_t2 = [V.at[i, "pair_id"] for i, k in zip(duoi.index, key) if k in k_t2]
    n_q = them_ly_do(drop_t2, "T2_CHUA_KIEM")
    cong("P10_tang2_Qwen", "tầng 2 (đuôi dài hiếm)", "khong_can",
         f"phương án A: {so(n_q)} cặp gắn T2_CHUA_KIEM (Qwen không đạt: precision ~30% trên 100 cặp hiệu chỉnh)")
elif kq2.exists() and hc.exists():
    q = pd.read_csv(kq2, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    h = pd.read_csv(hc, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    # qwen_nhan của 300 cặp hiệu chỉnh lấy từ file kết quả Qwen (Claude gắn claude_nhan trước, không nhìn nhãn Qwen)
    q_map = dict(zip(zip(q.gia_tri_zh, q.gia_tri_vi), q.nhan_qwen))
    if "qwen_nhan" not in h.columns:
        h["qwen_nhan"] = ""
    h["qwen_nhan"] = [qn.strip() or q_map.get((z, v), "") for qn, z, v in zip(h.qwen_nhan, h.gia_tri_zh, h.gia_tri_vi)]
    h = h[(h.claude_nhan.str.strip() != "") & (h.qwen_nhan.str.strip() != "")]
    # Đo trên đợt giữ riêng: đợt 1 (mau100_02-10) đã dùng để chọn ngưỡng trên Colab nên không dùng để đo
    if "dot" in h.columns and (h["dot"] == "mau200_03-10").any():
        h = h[h["dot"] == "mau200_03-10"]
    co = h.qwen_nhan.str.upper().eq("SAI_NGHIA")
    that_sai = h.claude_nhan.str.upper().eq("SAI")
    prec = (that_sai[co]).mean() if co.any() else 0
    recall = (co[that_sai]).mean() if that_sai.any() else 0
    du_co = co.sum() >= 20   # quá ít cờ thì độ chính xác không tin được
    cong("P10_tang2_Qwen", "độ chính xác cờ SAI_NGHIA của Qwen", "dat" if (prec >= 0.8 and du_co) else "khong_dat",
         f"precision = {prec:.0%} trên {co.sum()} cờ (cần ≥ 80% và ≥ 20 cờ); recall = {recall:.0%} trên {that_sai.sum()} cặp Claude gắn SAI; {len(h)} cặp hiệu chỉnh")
    if prec >= 0.8 and du_co:
        sai_q = set(zip(q.gia_tri_zh[q.nhan_qwen.str.upper() == "SAI_NGHIA"], q.gia_tri_vi[q.nhan_qwen.str.upper() == "SAI_NGHIA"]))
        drop_q = [V.at[i, "pair_id"] for i, k in zip(duoi.index, key) if k in sai_q]
        n_q = them_ly_do(drop_q, "QWEN_SAI")
else:
    thieu = [f.name for f in (kq2, hc) if not f.exists()]
    cong("P10_tang2_Qwen", "Qwen sàng lọc tầng 2 trên Colab", "chua_chay",
         f"chưa có {', '.join(thieu)} (chạy notebooks/desc-qwen-tang2-colab.ipynb trên Colab)")
da_kiem = {k for k, (nh, _) in nhan1.items() if nh in ("DUNG", "SAI")}
chua_kiem = [V.at[i, "pair_id"] for i, k in zip(duoi.index, key) if k not in da_kiem]
them_nhan(set(chua_kiem) | set(kc), "chua_kiem_nghia")
luu("p10")
ket_luan(f"Tầng 1: {so(len(tang1))} cặp khác nhau (gồm {so(tang1.bat_thuong.sum())} cặp độ dài bất thường); sửa {so(n10s)} giá trị, loại {so(len(set(sai_drop)))} cặp.",
         f"Tầng 2: loại {so(n_q)} cặp ({'mã T2_CHUA_KIEM, phương án A' if not DUNG_QWEN else 'mã QWEN_SAI'}); "
         f"nhãn chua_kiem_nghia: {so(P.nhan.str.contains('chua_kiem_nghia').sum())} cặp.")

## P11. Loại có mã lý do

Kiểm lần cuối trên cặp chưa bị loại:
- Tên hoặc giá trị tiếng Việt còn chữ Hán (trừ cặp mã/giấy tờ) thì gắn `DS07`.
- Sản phẩm có mã `DS02` thì loại cả sản phẩm.

`dung_cho_train` = 1 khi cặp không có mã lý do nào. Mã lý do khác với nhãn: nhãn như `it_thong_tin`, `chua_kiem_nghia`, `vai_tro_tam` chỉ để biết, không loại cặp.

In [ ]:
vai = P.set_index("pair_id").vai_tro
con_cjk_v = V[V.v_vi.map(has_cjk) & ~V.la_ma].pair_id
con_cjk_n = P.loc[P.name_vi.map(has_cjk), "pair_id"]
n07 = them_ly_do(set(con_cjk_v) | set(con_cjk_n), "DS07")
P["dung_cho_train"] = (P.ly_do == "").astype(int)
ma = Counter(c for s in P.ly_do for c in s.split(";") if c)
print("Số cặp theo mã lý do (một cặp có thể nhiều mã):")
display(pd.Series(ma).sort_values(ascending=False).rename("so_cap").to_frame().T)
theo_cat = P.groupby("category").agg(so_cap=("pair_id", "size"), giu=("dung_cho_train", "sum"))
theo_cat["pct_loai"] = (100 * (1 - theo_cat.giu / theo_cat.so_cap)).round(1)
print("Tỉ lệ cặp bị loại theo category:")
display(theo_cat)
luu("p11")
ket_luan(f"DS07 (còn chữ Hán sau P6/P7): {so(n07)} cặp.",
         f"Giữ cho train {so(P.dung_cho_train.sum())}/{so(len(P))} cặp ({tl(P.dung_cho_train.sum(), len(P))}); bị loại nhiều nhất ở category: "
         + ", ".join(f"{c} {v}%" for c, v in theo_cat.pct_loai.sort_values(ascending=False).head(3).items()) + ".")

## P12. Khử trùng lặp và gắn nhóm (DS11)

- **Trùng y hệt:** sản phẩm có description sạch (cả zh lẫn vi) trùng y hệt sản phẩm khác thì gắn `trung_y_het` = 1 cho bản thứ hai trở đi.
- **Nhóm trùng (`nhom_trung`):** gộp các sản phẩm có cùng title_zh, cùng shop, cùng description, hoặc gần trùng theo MinHash (Jaccard ≥ 0,8).
- Chia train/dev/test **đã hoãn**; nhóm này chỉ để dùng sau.

In [ ]:
Pk = P[P.dung_cho_train == 1].sort_values(["row", "vi_tri"])
gv = V.groupby("pair_id").agg(vz=("v_zh", ", ".join), vv=("v_vi", ", ".join))
Pk = Pk.join(gv, on="pair_id")
Pk["seg_zh"] = Pk.name_zh + ": " + Pk.vz.fillna("")
Pk["seg_vi"] = Pk.name_vi + ": " + Pk.vv.fillna("")
desc = Pk.groupby("row").agg(description_zh_sach=("seg_zh", "; ".join), description_vi_sach=("seg_vi", "; ".join),
                             so_cap_giu=("pair_id", "size"))
SP = SP.drop(columns=[c for c in ["description_zh_sach", "description_vi_sach", "so_cap_giu"] if c in SP]).join(desc)
SP["description_zh_sach"] = SP.description_zh_sach.fillna("")
SP["description_vi_sach"] = SP.description_vi_sach.fillna("")
SP["so_cap_giu"] = SP.so_cap_giu.fillna(0).astype(int)
SP["so_cap_loai"] = P.groupby("row").dung_cho_train.apply(lambda s: int((s == 0).sum())).reindex(SP.index).fillna(0).astype(int)
co_desc = (SP.ly_do_sp == "") & (SP.so_cap_giu > 0)
SP["trung_y_het"] = (co_desc & SP.duplicated(["description_zh_sach", "description_vi_sach"], keep="first")).astype(int)

parent = list(range(len(SP)))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb


for col in ["title_zh", "shop", "description_zh_sach"]:
    first = {}
    for i, v in enumerate(SP[col].fillna("")):
        if not v:
            continue
        if v in first:
            union(i, first[v])
        else:
            first[v] = i
mhs = []
lsh = MinHashLSH(threshold=0.8, num_perm=128)
for i, s in enumerate(SP.description_zh_sach):
    m = MinHash(num_perm=128, seed=SEED)
    m.update_batch([s[j:j + 5].encode("utf-8") for j in range(max(1, len(s) - 4))])
    mhs.append(m)
    lsh.insert(i, m)
for i, m in enumerate(mhs):
    for j in lsh.query(m):
        if j > i and m.jaccard(mhs[j]) >= 0.8:
            union(i, j)
SP["nhom_trung"] = [find(i) for i in range(len(SP))]
sz = SP.nhom_trung.value_counts()
SP.to_parquet(OUT / "p12_san_pham.parquet", compression="zstd")
ket_luan(f"Trùng y hệt sau khi sạch: {so(SP.trung_y_het.sum())} sản phẩm.",
         f"nhom_trung: {so(len(sz))} nhóm; nhóm lớn nhất {so(sz.iloc[0])} sản phẩm ({tl(sz.iloc[0], len(SP), 2)}).")

## P13. Dựng output và kiểm định

**Output** (trong `data/processed/desc/`):
- `desc_pairs_clean.parquet`: mọi cặp, kèm giá trị zh/vi đã sạch, mã lý do, nhãn, vai trò và cột `dung_cho_train`.
- `desc_clean.parquet`: mỗi sản phẩm một dòng, gồm description_zh/vi dựng lại từ các cặp giữ lại, `nhom_trung` và `trung_y_het`.
- `log_thay_doi.parquet`: mọi thay đổi theo từng bước.

**Kiểm định:**
- Đo lại các lỗi chính trên phần giữ lại, so trước và sau.
- Bảng trạng thái các cổng duyệt.
- 200 sản phẩm ngẫu nhiên để anh đọc bản trước và sau (`duyet/P13_mau_cuoi_mau_v3.csv`), chỉ bốc khi mọi cổng khác đã đạt.

In [ ]:
gv = V.groupby("pair_id").agg(gia_tri_zh=("v_zh", ", ".join), gia_tri_vi=("v_vi", ", ".join), gia_tri_vi_goc=("v_vi_goc", ", ".join))
OUTP = P.join(gv, on="pair_id")
OUTP.to_parquet(OUT / "desc_pairs_clean.parquet", compression="zstd")
SP.drop(columns=["description_zh", "description_vi"]).to_parquet(OUT / "desc_clean.parquet", compression="zstd")
LOGDF = pd.concat(LOG, ignore_index=True) if LOG else pd.DataFrame()
LOGDF.to_parquet(OUT / "log_thay_doi.parquet", compression="zstd")
print("Số thay đổi theo bước:")
display(LOGDF.groupby(["buoc", "cot"]).size().rename("so_thay_doi").to_frame().T)

# Đo lại trên phần giữ lại (trước = bản gốc của chính các cặp đó)
G = OUTP[OUTP.dung_cho_train == 1]
Vg = V[V.pair_id.isin(G.pair_id)]


def do(vv, names):
    nz = {}
    nz["giá trị còn chữ Hán (không tính mã)"] = int((vv.map(has_cjk) & ~Vg.la_ma.values).sum())
    nz["giá trị có số đuôi kiểu đánh số"] = int(sum(bool(re.search(r"(Không|Có|Là|Khác|KHÔNG|Đúng)\.?\d+$", s)) for s in vv))
    nz["giá trị có dấu chấm cuối do máy thêm"] = int(sum(s.endswith(".") and not s.endswith("...") and not z.rstrip().endswith(("。", "."))
                                                       and not re.search(r"(?:\b[A-Za-z]\.){2,}$", s) for s, z in zip(vv, Vg.v_zh)))
    nz["giá trị IN HOA toàn bộ"] = int(sum(in_hoa_viet(s) for s in vv))
    nz["giá trị có 【】"] = int(sum(("【" in s) or ("】" in s) for s in vv))
    nz["tên zh có > 1 cách dịch (đã bỏ chữ hoa)"] = int(names.groupby(G.name_zh.values).agg(lambda s: s.map(norm).nunique()).gt(1).sum())
    return nz


truoc = do(Vg.v_vi_goc.reset_index(drop=True), G.name_vi_goc.reset_index(drop=True))
sau = do(Vg.v_vi.reset_index(drop=True), G.name_vi.reset_index(drop=True))
display(pd.DataFrame({"truoc_khi_sach": truoc, "sau_khi_sach": sau}))

print("\nTrạng thái cổng duyệt:")
GT = pd.DataFrame(GATES)
display(GT)
chinh_thuc = GT.trang_thai.isin(["dat", "mac_dinh", "khong_can"]).all()
mau = SP[(SP.ly_do_sp == "") & (SP.so_cap_giu > 0)].assign(
    description_vi_goc=lambda d: d.description_vi.str[:600], description_vi_sach_cat=lambda d: d.description_vi_sach.str[:600])
# Mẫu cuối chỉ bốc khi MỌI cổng khác đã đạt: bốc sớm thì nội dung trong mẫu sẽ cũ khi từ điển/luật còn đổi.
# (v1, v2 là các lần bốc sớm trước đây, không còn được đọc.)
if chinh_thuc:
    mau_kiem("P13_mau_cuoi", mau, 200, ["product_id", "category", "description_vi_goc", "description_vi_sach_cat"],
             viec="200 sản phẩm trước/sau", phien="v3")
else:
    cong("P13_mau_cuoi", "200 sản phẩm trước/sau", "chua_toi", "bốc mẫu khi mọi cổng khác đã đạt")
# 03/10: tính lại sau khi có cổng P13 (trước đây tính trước nên báo "chính thức" khi P13 còn chờ kiểm)
GT = pd.DataFrame(GATES)
chinh_thuc = GT.trang_thai.isin(["dat", "mac_dinh", "khong_can"]).all()
print(f"\nTổng thời gian: {time.time() - T0:.0f} giây")
ket_luan(f"Giữ {so(len(G))}/{so(len(OUTP))} cặp ({tl(len(G), len(OUTP))}) cho train; {so((SP.so_cap_giu > 0).sum())} sản phẩm có description sạch.",
         "Dữ liệu là CHÍNH THỨC." if chinh_thuc else
         f"Dữ liệu CHƯA chính thức: còn {so((~GT.trang_thai.isin(['dat', 'mac_dinh', 'khong_can'])).sum())} cổng chưa đạt (bảng trên). Duyệt xong file nào thì chạy lại notebook.")